# 교통안전 EPDO 통합 분석 (STEP 01 ~ STEP 11)

COMPAS 환경 실행용 — `BASE_DIR = /data`

모든 STEP을 순서대로 실행하세요.  
각 STEP의 출력 CSV는 `/data/epdo_analysis/output/` 에 저장됩니다.

In [ ]:
import csv, json, math, os, warnings
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, roc_auc_score)
from sklearn.model_selection import train_test_split
import geopandas as gpd
from shapely.geometry import Point

try:
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except ImportError:
    from sklearn.linear_model import PoissonRegressor
    HAS_STATSMODELS = False

warnings.filterwarnings("ignore", category=UserWarning)

# ── 경로 설정 ──────────────────────────────────────────────────────────────────
# COMPAS 구조: 노트북과 같은 위치에 data/ 폴더가 있음
BASE_DIR   = os.getcwd()          # 노트북 실행 위치 (data/ 폴더의 부모)
EPDO_DIR   = os.getcwd()          # 분석 결과 저장 위치
OUTPUT_DIR = os.path.join(EPDO_DIR, "epdo_analysis", "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"BASE_DIR  : {BASE_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

---

## STEP 00 - 전처리 (원시 데이터 → 분석용 파일 생성)

원시 COMPAS 데이터로부터 STEP 01~11에 필요한 중간 파일을 생성합니다.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os, csv, warnings
from collections import defaultdict, Counter

warnings.filterwarnings("ignore")
CRS_GEO  = "EPSG:4326"
CRS_PROJ = "EPSG:5186"
DATA_DIR  = os.path.join(BASE_DIR, "data")

def find_col(df, *kws):
    cl = {c.lower().replace(" ","").replace("_",""): c for c in df.columns}
    for kw in kws:
        kw2 = kw.lower().replace(" ","").replace("_","")
        for k, v in cl.items():
            if kw2 in k: return v
    return None

def to_gdf(csv_path):
    df = pd.read_csv(csv_path, encoding="utf-8-sig", low_memory=False)
    lat = find_col(df, "위도","lat","latitude","ycoord","y좌표")
    lon = find_col(df, "경도","lon","longitude","xcoord","x좌표")
    if not lat or not lon:
        print(f"  좌표 탐지 실패: {os.path.basename(csv_path)}  컬럼: {list(df.columns[:8])}")
        return None
    df[lat] = pd.to_numeric(df[lat], errors="coerce")
    df[lon] = pd.to_numeric(df[lon], errors="coerce")
    df = df.dropna(subset=[lat, lon])
    src = CRS_PROJ if df[lat].median() > 1000 else CRS_GEO
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lon], df[lat]), crs=src)
    return gdf.to_crs(CRS_PROJ)

print("=" * 60)
print("STEP 00 - 원시 데이터 전처리")
print("=" * 60)

# 1. 기본 데이터 로드
print("\n[1] 격자 / 도로망 / 사고 데이터 로드...")
grids = gpd.read_file(os.path.join(DATA_DIR, "01._격자_(4개_시·구).geojson")).to_crs(CRS_PROJ)
gc = find_col(grids, "gid") or grids.columns[0]
grids = grids.rename(columns={gc: "gid"})
print(f"    격자: {len(grids)}개")

roads = gpd.read_file(os.path.join(DATA_DIR, "08.상세도로망_네트워크.geojson")).to_crs(CRS_PROJ)
print(f"    링크: {len(roads)}개  컬럼: {[c for c in roads.columns if c != 'geometry']}")

acc_raw = gpd.read_file(os.path.join(DATA_DIR, "13._교통사고이력.geojson"))
print(f"    사고: {len(acc_raw)}건  컬럼: {[c for c in acc_raw.columns if c != 'geometry']}")
acc = acc_raw.to_crs(CRS_PROJ)

uid_c  = find_col(acc, "uid","번호","사고번호") or acc.columns[0]
svr_c  = find_col(acc, "사상자","svrity","injury","피해구분","중증도","severity")
yr_c   = find_col(acc, "발생년","년도","acc_yr","year")
mon_c  = find_col(acc, "발생월","acc_mon","month")
time_c = find_col(acc, "발생시","시각","acc_time","hour","time")
week_c = find_col(acc, "요일","week")
type_c = find_col(acc, "사고유형","acc_type","type")
viol_c = find_col(acc, "법규위반","violation","viol")
print(f"    매핑: uid={uid_c}, 심각도={svr_c}, 년={yr_c}, 월={mon_c}, 요일={week_c}")

SVRITY_KEYS = ["사망","중상","경상","부상신고","상해없음","기타불명"]
WEEKEND = {"토","토요일","일","일요일","sat","sun","saturday","sunday"}

def norm_svr(v):
    v = str(v).strip()
    for k in SVRITY_KEYS:
        if k in v: return k
    return v

def norm_week(v):
    return "주말" if str(v).strip().lower() in WEEKEND else "평일"

# 2. 교통사고 → 링크 매핑
print("\n[2] 교통사고 → 링크 매핑 (sjoin_nearest)...")
acc_r = acc.reset_index(drop=True)
road_keep = [c for c in ["link_id","road_name","road_rank","geometry"] if c in roads.columns or c == "geometry"]
mapped = gpd.sjoin_nearest(acc_r, roads[road_keep].reset_index(drop=True), how="left", distance_col="_d")
mapped["_lon"] = acc_raw.reset_index(drop=True).geometry.x
mapped["_lat"] = acc_raw.reset_index(drop=True).geometry.y

def gv(row, col, default=""):
    return row[col] if col and col in row.index and pd.notna(row[col]) else default

out_link = []
for _, r in mapped.iterrows():
    out_link.append({
        "uid":           gv(r, uid_c),
        "link_id":       gv(r, "link_id"),
        "road_name":     gv(r, "road_name"),
        "injury_svrity": norm_svr(gv(r, svr_c)) if svr_c else "",
        "epdo_score":    0,
        "acc_yr":        gv(r, yr_c),
        "acc_mon":       gv(r, mon_c),
        "acc_time":      gv(r, time_c),
        "week_type":     norm_week(gv(r, week_c)) if week_c else "평일",
        "acc_type":      gv(r, type_c),
        "violation":     gv(r, viol_c),
        "road_type":     gv(r, "road_rank"),
        "lon":           r["_lon"],
        "lat":           r["_lat"],
    })

OUT_LINK = os.path.join(OUTPUT_DIR, "00_교통사고_링크매핑.csv")
with open(OUT_LINK, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(out_link[0].keys()))
    w.writeheader(); w.writerows(out_link)
print(f"    저장: {OUT_LINK} ({len(out_link):,}건)")
print(f"    심각도: {dict(Counter(r['injury_svrity'] for r in out_link).most_common(6))}")

# 3. 교통사고 → 격자 매핑
print("\n[3] 교통사고 → 격자 매핑...")
ag = gpd.sjoin(acc_r, grids[["gid","geometry"]], how="left", predicate="within")
ag_gid = ag["gid"].groupby(ag.index).first()  # deduplicate boundary points
acc_out = acc_raw.copy()
acc_out["grid_gid"] = ag_gid.reindex(acc_out.index).values
if svr_c:
    acc_out["injury_svrity"] = acc_out[svr_c].apply(norm_svr)
elif "injury_svrity" not in acc_out.columns:
    acc_out["injury_svrity"] = ""
OUT_GJ = os.path.join(OUTPUT_DIR, "00_교통사고_격자매핑.geojson")
acc_out.to_file(OUT_GJ, driver="GeoJSON")
print(f"    저장: {OUT_GJ}")

# 4. 격자별 인프라 통합
print("\n[4] 격자별 인프라 통합...")
res = pd.DataFrame({"grid_gid": grids["gid"].values})

FACILS = [
    ("18._횡단보도_위치정보.csv",  "crosswalk_cnt"),
    ("14._어린이보호구역.csv",      "child_zone_cnt"),
    ("15._학교현황.csv",            "school_cnt"),
    ("16._유치원현황.csv",          "kindergarten_cnt"),
    ("17._어린이집현황.csv",        "daycare_cnt"),
    ("19._버스정류장_위치정보.csv", "bus_stop_cnt"),
    ("20._CCTV_현황.csv",           "cctv_cnt"),
]
for fname, col in FACILS:
    fp = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fp):
        res[col] = 0; print(f"  없음: {fname}"); continue
    gdf = to_gdf(fp)
    if gdf is None:
        res[col] = 0; continue
    j = gpd.sjoin(gdf, grids[["gid","geometry"]], how="left", predicate="within")
    agg = j.groupby("gid").size().reset_index().rename(columns={"gid":"grid_gid", 0: col})
    res = res.merge(agg, on="grid_gid", how="left")
    res[col] = res[col].fillna(0).astype(int)
    print(f"  {fname}: {res[col].sum():.0f}개")

# CCTV 대수
fp = os.path.join(DATA_DIR, "20._CCTV_현황.csv")
if os.path.exists(fp):
    df_cc = pd.read_csv(fp, encoding="utf-8-sig", low_memory=False)
    cam_c = find_col(df_cc, "대수","카메라","설치대수","cam")
    gdf_cc = to_gdf(fp)
    if gdf_cc is not None and cam_c:
        gdf_cc[cam_c] = pd.to_numeric(gdf_cc[cam_c], errors="coerce").fillna(1)
        j = gpd.sjoin(gdf_cc[["geometry", cam_c]], grids[["gid","geometry"]], how="left", predicate="within")
        agg = j.groupby("gid")[cam_c].sum().reset_index().rename(columns={"gid":"grid_gid", cam_c:"cctv_cam_cnt"})
        res = res.merge(agg, on="grid_gid", how="left")
    else:
        res["cctv_cam_cnt"] = res["cctv_cnt"]
else:
    res["cctv_cam_cnt"] = res["cctv_cnt"]
res["cctv_cam_cnt"] = res["cctv_cam_cnt"].fillna(0).astype(int)

# 유치원 원아수
fp = os.path.join(DATA_DIR, "16._유치원현황.csv")
if os.path.exists(fp):
    df_kg = pd.read_csv(fp, encoding="utf-8-sig", low_memory=False)
    cc = find_col(df_kg, "원아수","학생수","아동수","정원","child","student")
    gdf_kg = to_gdf(fp)
    if gdf_kg is not None and cc:
        gdf_kg[cc] = pd.to_numeric(gdf_kg[cc], errors="coerce").fillna(0)
        j = gpd.sjoin(gdf_kg[["geometry", cc]], grids[["gid","geometry"]], how="left", predicate="within")
        agg = j.groupby("gid")[cc].sum().reset_index().rename(columns={"gid":"grid_gid", cc:"kindergarten_child_cnt"})
        res = res.merge(agg, on="grid_gid", how="left")
    else:
        res["kindergarten_child_cnt"] = res["kindergarten_cnt"]
else:
    res["kindergarten_child_cnt"] = res["kindergarten_cnt"]
res["kindergarten_child_cnt"] = res["kindergarten_child_cnt"].fillna(0).astype(int)

# 과속방지턱 (포인트 파일 → sjoin으로 격자 집계)
fp = os.path.join(DATA_DIR, "21.1_과속방지턱_격자매핑.csv")
if os.path.exists(fp):
    df_sb = pd.read_csv(fp, encoding="utf-8-sig", low_memory=False)
    print(f"  과속방지턱 컬럼: {list(df_sb.columns)}")
    hgt_c = find_col(df_sb, "fac_hght","height","hght","높이")
    gdf_sb = to_gdf(fp)
    if gdf_sb is not None:
        if hgt_c:
            gdf_sb[hgt_c] = pd.to_numeric(gdf_sb[hgt_c], errors="coerce").fillna(0)
            gdf_sb["_hght_below"] = gdf_sb[hgt_c].apply(lambda x: x if x <= 10 else 0)
            gdf_sb["_hght_above"] = gdf_sb[hgt_c].apply(lambda x: x if x > 10 else 0)
        else:
            gdf_sb["_hght_below"] = gdf_sb["_hght_above"] = 0
        gdf_sb["_cnt"] = 1
        j = gpd.sjoin(gdf_sb[["geometry","_cnt","_hght_below","_hght_above"]], grids[["gid","geometry"]], how="left", predicate="within")
        agg_sb = j.groupby("gid")[["_cnt","_hght_below","_hght_above"]].sum().reset_index()
        agg_sb = agg_sb.rename(columns={"gid":"grid_gid","_cnt":"speedbump_cnt","_hght_below":"speedbump_hght_below","_hght_above":"speedbump_hght_above"})
        res = res.merge(agg_sb, on="grid_gid", how="left")
        print(f"  과속방지턱: {int(res['speedbump_cnt'].sum())}개")
    else:
        res["speedbump_cnt"] = res["speedbump_hght_below"] = res["speedbump_hght_above"] = 0
else:
    res["speedbump_cnt"] = res["speedbump_hght_below"] = res["speedbump_hght_above"] = 0
for c in ["speedbump_cnt","speedbump_hght_below","speedbump_hght_above"]:
    res[c] = res[c].fillna(0).astype(int)

OUT_INFRA = os.path.join(OUTPUT_DIR, "00_격자별_통합데이터.csv")
res.to_csv(OUT_INFRA, index=False, encoding="utf-8-sig")
print(f"\n    저장: {OUT_INFRA}  ({len(res):,}개 격자)")

# 5. 링크별 사고 집계
print("\n[5] 링크별 사고 집계...")
link_agg = defaultdict(list)
for r in out_link:
    lid = str(r.get("link_id",""))
    if lid and lid not in ("","nan"):
        link_agg[lid].append(str(r.get("uid","")))

OUT_AGG = os.path.join(OUTPUT_DIR, "00_링크별_사고집계.csv")
with open(OUT_AGG, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f)
    w.writerow(["link_id","accident_cnt","accident_uids"])
    for lid, uids in link_agg.items():
        w.writerow([lid, len(uids), "|".join(uids)])
print(f"    저장: {OUT_AGG} ({len(link_agg):,}개 링크)")

print("\n" + "="*60)
print("STEP 00 완료")
print("="*60)


---

## 01_EPDO_SCORE

# STEP 01 - 사고별 EPDO 점수 부여

- 입력: output/13._교통사고_링크매핑.csv
- 출력: epdo_analysis/output/01_사고별_EPDO점수.csv

EPDO 가중치 출처:
  이상엽(2019). 교통사고 원인행위의 벌점 추정에 관한 연구.
  대한교통학회지, 37(5), 365-378.
  사망=391, 중상=69, 경상=8, 부상신고=6, 물적피해(PDO)=1

In [ ]:
import csv
import os

INPUT_PATH  = os.path.join(EPDO_DIR, "epdo_analysis", "output", "00_교통사고_링크매핑.csv")
OUTPUT_PATH = os.path.join(EPDO_DIR, "epdo_analysis", "output", "01_사고별_EPDO점수.csv")

EPDO_WEIGHTS = {
    "사망":   391,
    "중상":    69,
    "경상":     8,
    "부상신고":  6,
    "상해없음":  1,
    "기타불명":  8,   # 경상과 동일 (보수적 처리)
    "":          0,   # 차량단독·피해자 없음
}

In [ ]:
print("=" * 60)
print("STEP 01 - 사고별 EPDO 점수 부여")
print("=" * 60)

with open(INPUT_PATH, "r", encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))

print(f"\n입력 사고 수: {len(rows):,}건")

result = []
unknown = {}
for row in rows:
    svrity = row.get("injury_svrity", "").strip()
    score = EPDO_WEIGHTS.get(svrity)
    if score is None:
        unknown[svrity] = unknown.get(svrity, 0) + 1
        score = 0
    result.append({
        "uid":           row["uid"],
        "link_id":       row["link_id"],
        "road_name":     row["road_name"],
        "injury_svrity": svrity,
        "epdo_score":    score,
        "acc_yr":        row["acc_yr"],
        "acc_mon":       row["acc_mon"],
        "acc_time":      row["acc_time"],
        "week_type":     row["week_type"],
        "acc_type":      row["acc_type"],
        "violation":     row["violation"],
        "road_type":     row.get("road_rank", ""),
        "lon":           row["lon"],
        "lat":           row["lat"],
    })

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
cols = list(result[0].keys())
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(result)

# 검증 출력
from collections import Counter
dist = Counter(r["injury_svrity"] for r in result)
epdo_by_svrity = {}
for r in result:
    k = r["injury_svrity"]
    epdo_by_svrity[k] = epdo_by_svrity.get(k, 0) + r["epdo_score"]

total_epdo = sum(r["epdo_score"] for r in result)

print(f"\n[심각도별 건수 및 EPDO 합계]")
for k, cnt in dist.most_common():
    print(f"  {k or '(빈값)':10s}: {cnt:5d}건 | EPDO {epdo_by_svrity.get(k,0):,}점")

print(f"\n전체 EPDO 합계: {total_epdo:,}점")
print(f"저장: {OUTPUT_PATH}")

if unknown:
    print(f"\n⚠ 미정의 값(0점 처리): {unknown}")

print("=" * 60)

---

## 02_LINK_TRAFFIC

# STEP 02 - 링크별 교통량 집계

- 입력: data/08.상세도로망_네트워크.geojson (up_v_link, dw_v_link → link_id 매핑)
        data/10._추정교통량.csv (v_link_id, ALL_AADT, k_length)
- 출력: epdo_analysis/output/02_링크별_교통량.csv

노출량(exposure) = ALL_AADT_total × k_length (대·km/일)

In [ ]:
import csv
import json
import os
from collections import defaultdict

ROAD_FILE   = os.path.join(BASE_DIR, "data", "08.상세도로망_네트워크.geojson")
TRAFFIC_FILE = os.path.join(BASE_DIR, "data", "10._추정교통량.csv")
OUTPUT_PATH = os.path.join(EPDO_DIR, "epdo_analysis", "output", "02_링크별_교통량.csv")

In [ ]:
print("=" * 60)
print("STEP 02 - 링크별 교통량 집계")
print("=" * 60)

# 1. 08번에서 v_link_id → link_id 역매핑
print("\n[1] 도로망 로드 중...")
with open(ROAD_FILE, "r", encoding="utf-8") as f:
    road_data = json.load(f)

vlink_to_link = {}   # v_link_id(str) → link_id(str)
link_props    = {}   # link_id(str) → {road_name, road_rank, length, k_length}
for feat in road_data["features"]:
    p = feat["properties"]
    lid = str(p["link_id"])
    link_props[lid] = {
        "road_name": p.get("road_name", ""),
        "road_rank": p.get("road_rank", ""),
        "length":    p.get("length", ""),
    }
    up = p.get("up_v_link")
    dw = p.get("dw_v_link")
    if up:
        vlink_to_link[str(up)] = lid
    if dw:
        vlink_to_link[str(dw)] = lid

print(f"    링크 수: {len(link_props):,}")
print(f"    v_link 매핑 수: {len(vlink_to_link):,}")

# 2. 10번 교통량 집계 (timeslot별 ALL_AADT 합산)
print("\n[2] 교통량 데이터 로드 중...")
aadt_by_vlink  = defaultdict(float)
klen_by_vlink  = {}
rank_by_vlink  = {}

with open(TRAFFIC_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        vid = row["v_link_id"]
        try:
            aadt_by_vlink[vid] += float(row["ALL_AADT"])
        except (ValueError, KeyError):
            pass
        klen_by_vlink[vid]  = row.get("k_length", "")
        rank_by_vlink[vid]  = row.get("road_rank", "")

print(f"    v_link_id 수: {len(aadt_by_vlink):,}")

# 3. v_link_id → link_id 변환 후 상행/하행 평균
link_aadt = defaultdict(list)   # link_id → [aadt_values]
link_klen = {}

for vid, aadt in aadt_by_vlink.items():
    lid = vlink_to_link.get(vid)
    if lid:
        link_aadt[lid].append(aadt)
        if lid not in link_klen:
            link_klen[lid] = klen_by_vlink.get(vid, "")

print(f"    link_id 매칭 수: {len(link_aadt):,} / {len(link_props):,}")

# 4. 링크별 교통량 산출 (상행·하행 평균)
result = []
for lid, aadt_list in link_aadt.items():
    all_aadt_total = sum(aadt_list) / len(aadt_list)   # 방향 평균
    try:
        k_length = float(link_klen.get(lid, 0))
    except ValueError:
        k_length = 0.0
    exposure = all_aadt_total * k_length   # 대·km/일

    props = link_props.get(lid, {})
    result.append({
        "link_id":       lid,
        "road_name":     props.get("road_name", ""),
        "road_rank":     props.get("road_rank", ""),
        "length_km":     props.get("length", ""),
        "k_length":      k_length,
        "ALL_AADT_total": round(all_aadt_total, 2),
        "exposure":      round(exposure, 4),
    })

result.sort(key=lambda x: -x["exposure"])

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
cols = list(result[0].keys())
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(result)

print(f"\n[결과] 교통량 산출 링크: {len(result):,}개")
print(f"       노출량 상위 5개:")
for r in result[:5]:
    print(f"  link_id={r['link_id']} | {r['road_name']} | AADT={r['ALL_AADT_total']:,.0f} | 노출량={r['exposure']:,.1f}")
print(f"\n저장: {OUTPUT_PATH}")
print("=" * 60)

---

## 03_LINK_EPDO_RISK

# STEP 03 - 링크별 위험도 산출

- 입력: epdo_analysis/output/01_사고별_EPDO점수.csv
        epdo_analysis/output/02_링크별_교통량.csv
        output/13._링크별_사고집계.csv (accident_uids 참조)
- 출력: epdo_analysis/output/03_링크별_위험도.csv

위험도(epdo_rate) = epdo_total / exposure × 1,000,000
  (단위: 백만 대·km당 EPDO, 교통량 미매칭 링크는 null)

In [ ]:
import csv
import os
from collections import defaultdict

EPDO_FILE     = os.path.join(EPDO_DIR, "epdo_analysis", "output", "01_사고별_EPDO점수.csv")
TRAFFIC_FILE  = os.path.join(EPDO_DIR, "epdo_analysis", "output", "02_링크별_교통량.csv")
AGG_FILE      = os.path.join(EPDO_DIR, "epdo_analysis", "output", "00_링크별_사고집계.csv")
OUTPUT_PATH   = os.path.join(EPDO_DIR, "epdo_analysis", "output", "03_링크별_위험도.csv")

In [ ]:
print("=" * 60)
print("STEP 03 - 링크별 위험도 산출")
print("=" * 60)

# 1. 사고별 EPDO 집계
print("\n[1] 사고별 EPDO 로드 중...")
link_epdo    = defaultdict(float)
link_cnt     = defaultdict(int)
link_meta    = {}   # link_id → {road_name, road_rank, max_speed, length}

with open(EPDO_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        lid = str(row["link_id"])
        link_epdo[lid] += float(row["epdo_score"])
        link_cnt[lid]  += 1
        if lid not in link_meta:
            link_meta[lid] = {
                "road_name": row["road_name"],
                "road_rank": row["road_type"],   # road_type에 road_rank 저장됨
            }

print(f"    사고 매핑 링크 수: {len(link_epdo):,}")

# 2. 교통량 로드
print("\n[2] 교통량 로드 중...")
traffic = {}
with open(TRAFFIC_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        lid = str(row["link_id"])
        traffic[lid] = {
            "ALL_AADT_total": float(row["ALL_AADT_total"]),
            "exposure":       float(row["exposure"]),
            "k_length":       row["k_length"],
            "length_km":      row["length_km"],
            "road_rank":      row["road_rank"],
            "road_name":      row["road_name"],
        }

# 3. accident_uids 로드
uids_by_link = {}
with open(AGG_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        uids_by_link[str(row["link_id"])] = row.get("accident_uids", "")

# 4. 링크별 위험도 산출
result = []
matched = 0
for lid, epdo_total in link_epdo.items():
    acc_cnt   = link_cnt[lid]
    epdo_avg  = round(epdo_total / acc_cnt, 4) if acc_cnt else 0
    meta      = link_meta.get(lid, {})
    traf      = traffic.get(lid)

    if traf and traf["exposure"] > 0:
        epdo_rate = round(epdo_total / traf["exposure"] * 1_000_000, 4)
        matched += 1
    else:
        epdo_rate = None

    result.append({
        "link_id":        lid,
        "road_name":      traf["road_name"] if traf else meta.get("road_name", ""),
        "road_rank":      traf["road_rank"] if traf else meta.get("road_rank", ""),
        "length_km":      traf["length_km"] if traf else "",
        "accident_cnt":   acc_cnt,
        "epdo_total":     round(epdo_total, 2),
        "epdo_avg":       epdo_avg,
        "ALL_AADT_total": round(traf["ALL_AADT_total"], 2) if traf else "",
        "exposure":       round(traf["exposure"], 4) if traf else "",
        "epdo_rate":      epdo_rate,
        "accident_uids":  uids_by_link.get(lid, ""),
    })

# epdo_rate 기준 정렬 (null은 후순위)
result.sort(key=lambda x: (x["epdo_rate"] is None, -(x["epdo_rate"] or 0)))

# 순위 부여 (epdo_rate 있는 것만)
rank = 1
for r in result:
    if r["epdo_rate"] is not None:
        r["epdo_rank"] = rank
        rank += 1
    else:
        r["epdo_rank"] = ""

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
cols = ["link_id", "road_name", "road_rank", "length_km",
        "accident_cnt", "epdo_total", "epdo_avg",
        "ALL_AADT_total", "exposure", "epdo_rate", "epdo_rank",
        "accident_uids"]
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(result)

print(f"\n[결과]")
print(f"    사고 발생 링크: {len(result):,}개")
print(f"    교통량 매칭 링크: {matched:,}개 ({matched/len(result)*100:.1f}%)")
print(f"\n    epdo_rate 상위 10개 (백만 대·km당 EPDO):")
for r in result[:10]:
    print(f"  rank={r['epdo_rank']:>4} | {r['road_name']:20s} | "
          f"사고{r['accident_cnt']:3d}건 | epdo_total={r['epdo_total']:>8.1f} | "
          f"rate={r['epdo_rate']:>10.2f}")
print(f"\n저장: {OUTPUT_PATH}")
print("=" * 60)

---

## 04_CAUSE_ANALYSIS

# STEP 04 - 위험 도로 원인 분석

- 입력: epdo_analysis/output/01_사고별_EPDO점수.csv
        epdo_analysis/output/03_링크별_위험도.csv
- 출력: epdo_analysis/output/04_위험도로_원인분석.csv

선정 기준: epdo_rate 기준 상위 20개 (사고 3건 이상 링크만 대상)
  - 사고 1~2건 링크는 노출량이 작아 rate가 극단적으로 높아지므로 제외
분석 항목: violation, acc_type, road_type(road_rank), acc_time, week_type

In [ ]:
import csv
import os
from collections import Counter, defaultdict

EPDO_FILE    = os.path.join(EPDO_DIR, "epdo_analysis", "output", "01_사고별_EPDO점수.csv")
RISK_FILE    = os.path.join(EPDO_DIR, "epdo_analysis", "output", "03_링크별_위험도.csv")
OUTPUT_PATH  = os.path.join(EPDO_DIR, "epdo_analysis", "output", "04_위험도로_원인분석.csv")

MIN_ACCIDENT = 3    # 최소 사고 건수 기준
TOP_N        = 20   # 분석 대상 상위 링크 수


def top_item(counter, n=3):
    return " | ".join(f"{k}({v}건)" for k, v in counter.most_common(n))

print("=" * 60)
print("STEP 04 - 위험 도로 원인 분석")
print("=" * 60)

# 1. 위험도 로드 → 상위 링크 선정
print(f"\n[1] 위험도 로드 중 (사고 {MIN_ACCIDENT}건 이상, 상위 {TOP_N}개)...")
risk_rows = []
with open(RISK_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        if row["epdo_rate"] and int(row["accident_cnt"]) >= MIN_ACCIDENT:
            risk_rows.append(row)

risk_rows.sort(key=lambda x: -float(x["epdo_rate"]))
top_links = risk_rows[:TOP_N]
top_link_ids = {r["link_id"] for r in top_links}
print(f"    대상 링크: {len(top_links)}개")

# 2. 사고 데이터 로드
print("\n[2] 사고 데이터 로드 중...")
accidents_by_link = defaultdict(list)
all_accidents = []

with open(EPDO_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        all_accidents.append(row)
        if row["link_id"] in top_link_ids:
            accidents_by_link[row["link_id"]].append(row)

# 3. 전체 평균 분포 (비교 기준)
all_violation = Counter(r["violation"] for r in all_accidents)
all_acc_type  = Counter(r["acc_type"]  for r in all_accidents)
all_time      = Counter(r["acc_time"]  for r in all_accidents)
all_week      = Counter(r["week_type"] for r in all_accidents)
total_all     = len(all_accidents)

# 4. 링크별 원인 분석
print("\n[3] 원인 분석 중...")
result = []

for r in top_links:
    lid  = r["link_id"]
    accs = accidents_by_link[lid]
    cnt  = len(accs)

    v_cnt   = Counter(a["violation"] for a in accs)
    t_cnt   = Counter(a["acc_type"]  for a in accs)
    time_cnt = Counter(a["acc_time"] for a in accs)
    week_cnt = Counter(a["week_type"] for a in accs)
    svr_cnt  = Counter(a["injury_svrity"] for a in accs)

    # 주말 비율
    weekend_ratio = round(week_cnt.get("주말", 0) / cnt * 100, 1) if cnt else 0

    # 전체 대비 특이 위반 (비율 차이)
    top_v = v_cnt.most_common(1)[0][0] if v_cnt else ""
    top_v_ratio = round(v_cnt.get(top_v, 0) / cnt * 100, 1)
    all_v_ratio = round(all_violation.get(top_v, 0) / total_all * 100, 1)

    result.append({
        "epdo_rank":         r["epdo_rank"],
        "link_id":           lid,
        "road_name":         r["road_name"],
        "road_rank":         r["road_rank"],
        "accident_cnt":      cnt,
        "epdo_total":        r["epdo_total"],
        "epdo_rate":         r["epdo_rate"],
        # 원인 분석
        "top_violation":     top_item(v_cnt, 3),
        "top_violation_1st": top_v,
        "top_v_ratio_pct":   top_v_ratio,
        "all_v_ratio_pct":   all_v_ratio,   # 전체 평균 비율
        "top_acc_type":      top_item(t_cnt, 2),
        "peak_time":         top_item(time_cnt, 3),
        "weekend_ratio_pct": weekend_ratio,
        "severity_dist":     top_item(svr_cnt, 4),
    })

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
cols = list(result[0].keys())
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(result)

print(f"\n[결과] 위험 도로 {len(result)}개 원인 분석 완료")
print(f"\n{'순위':>4} {'도로명':20s} {'사고':>4} {'epdo_rate':>12} {'주요위반'}")
print("-" * 80)
for r in result[:10]:
    print(f"  {r['epdo_rank']:>3} {r['road_name']:20s} {r['accident_cnt']:>4}건 "
          f"{float(r['epdo_rate']):>12.1f}  {r['top_violation_1st']}")

print(f"\n저장: {OUTPUT_PATH}")
print("=" * 60)

---

## 05_GRID_EPDO_INFRA

# STEP 05 - 격자별 EPDO + 인프라 통합 분석

- 입력: output/13._교통사고_격자매핑.geojson (사고별 grid_gid + injury_svrity)
        output/격자별_통합데이터.csv (grid_gid별 인프라 집계)
- 출력: epdo_analysis/output/05_격자별_EPDO_인프라통합.csv

EPDO 가중치 출처:
  이상엽(2019). 교통사고 원인행위의 벌점 추정에 관한 연구.
  대한교통학회지, 37(5), 365-378.

In [ ]:
import csv
import json
import os
from collections import defaultdict

ACCIDENT_FILE = os.path.join(EPDO_DIR, "epdo_analysis", "output", "00_교통사고_격자매핑.geojson")
INFRA_FILE   = os.path.join(EPDO_DIR, "epdo_analysis", "output", "00_격자별_통합데이터.csv")
OUTPUT_PATH  = os.path.join(EPDO_DIR, "epdo_analysis", "output", "05_격자별_EPDO_인프라통합.csv")

EPDO_WEIGHTS = {
    "사망":    391,
    "중상":     69,
    "경상":      8,
    "부상신고":   6,
    "상해없음":   1,
    "기타불명":   8,
    "":          0,
}

SEVERITY_KEYS = ["사망", "중상", "경상", "부상신고", "상해없음", "기타불명"]

In [ ]:
print("=" * 60)
print("STEP 05 - 격자별 EPDO + 인프라 통합 분석")
print("=" * 60)

# 1. 사고 데이터 로드 → 격자별 EPDO 집계
print("\n[1] 사고 데이터 로드 중...")
with open(ACCIDENT_FILE, "r", encoding="utf-8") as f:
    accident_data = json.load(f)

grid_epdo = defaultdict(lambda: {
    "epdo_total":  0.0,
    "accident_cnt": 0,
    **{f"{k}_cnt": 0 for k in SEVERITY_KEYS},
})

for feat in accident_data["features"]:
    props = feat["properties"]
    gid   = props.get("grid_gid", "")
    if not gid:
        continue
    svrity = (props.get("injury_svrity") or "").strip()
    score  = EPDO_WEIGHTS.get(svrity, 0)

    grid_epdo[gid]["epdo_total"]   += score
    grid_epdo[gid]["accident_cnt"] += 1
    key = f"{svrity}_cnt" if svrity in SEVERITY_KEYS else ""
    if key:
        grid_epdo[gid][key] += 1

print(f"    사고 격자 수: {len(grid_epdo):,}개 / 전체 사고: {len(accident_data['features']):,}건")

# 2. 인프라 데이터 로드
print("\n[2] 인프라 데이터 로드 중...")
infra = {}
with open(INFRA_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        infra[row["grid_gid"]] = row

print(f"    인프라 격자 수: {len(infra):,}개")

# 3. 통합
result = []
matched = 0
for gid, epdo in grid_epdo.items():
    epdo_total  = round(epdo["epdo_total"], 2)
    acc_cnt     = epdo["accident_cnt"]
    epdo_avg    = round(epdo_total / acc_cnt, 4) if acc_cnt else 0

    inf = infra.get(gid, {})
    if inf:
        matched += 1

    result.append({
        "grid_gid":          gid,
        "epdo_total":        epdo_total,
        "epdo_avg":          epdo_avg,
        "accident_cnt":      acc_cnt,
        "사망_cnt":           epdo["사망_cnt"],
        "중상_cnt":           epdo["중상_cnt"],
        "경상_cnt":           epdo["경상_cnt"],
        "부상신고_cnt":        epdo["부상신고_cnt"],
        "상해없음_cnt":        epdo["상해없음_cnt"],
        "기타불명_cnt":        epdo["기타불명_cnt"],
        "crosswalk_cnt":     inf.get("crosswalk_cnt", 0),
        "child_zone_cnt":    inf.get("child_zone_cnt", 0),
        "school_cnt":        inf.get("school_cnt", 0),
        "kindergarten_cnt":  inf.get("kindergarten_cnt", 0),
        "daycare_cnt":       inf.get("daycare_cnt", 0),
        "bus_stop_cnt":      inf.get("bus_stop_cnt", 0),
        "cctv_cnt":          inf.get("cctv_cnt", 0),
        "cctv_cam_cnt":      inf.get("cctv_cam_cnt", 0),
        "speedbump_cnt":     inf.get("speedbump_cnt", 0),
    })

result.sort(key=lambda x: -x["epdo_total"])

# 4. 저장
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
cols = list(result[0].keys())
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(result)

total_epdo = sum(r["epdo_total"] for r in result)
print(f"\n[결과]")
print(f"    격자 수: {len(result):,}개 | 인프라 매칭: {matched:,}개")
print(f"    전체 EPDO 합계: {total_epdo:,.0f}점")
print(f"\n    epdo_total 상위 10개:")
print(f"  {'격자ID':12s} {'EPDO':>8} {'사고':>4} {'사망':>3} {'중상':>4} {'횡단보도':>6} {'CCTV':>5} {'과속방지턱':>7}")
print("  " + "-" * 65)
for r in result[:10]:
    print(f"  {r['grid_gid']:12s} {r['epdo_total']:>8.0f} {r['accident_cnt']:>4} "
          f"{r['사망_cnt']:>3} {r['중상_cnt']:>4} {r['crosswalk_cnt']:>6} "
          f"{r['cctv_cnt']:>5} {r['speedbump_cnt']:>7}")

print(f"\n저장: {OUTPUT_PATH}")
print("=" * 60)

---

## 06_INFRA_ANALYSIS

# STEP 06 - 인프라 공백 분석

- 입력: epdo_analysis/output/05_격자별_EPDO_인프라통합.csv
- 출력: epdo_analysis/output/06_인프라_상관분석.csv
        epdo_analysis/output/06_인프라공백_고위험격자.csv

분석 1: EPDO × 시설물 수 상관관계 (Pearson + Spearman)
  - 어떤 시설물이 EPDO와 관련 있는지 확인
분석 2: 고위험 격자(상위 25%) 중 시설물 공백 격자 추출
  - 시설물이 없거나 전체 평균의 50% 미만인 항목을 '공백'으로 판단

In [ ]:
import csv
import math
import os

INPUT_PATH   = os.path.join(EPDO_DIR, "epdo_analysis", "output", "05_격자별_EPDO_인프라통합.csv")
CORR_OUTPUT  = os.path.join(EPDO_DIR, "epdo_analysis", "output", "06_인프라_상관분석.csv")
GAP_OUTPUT   = os.path.join(EPDO_DIR, "epdo_analysis", "output", "06_인프라공백_고위험격자.csv")

INFRA_COLS = [
    "crosswalk_cnt", "child_zone_cnt", "school_cnt",
    "kindergarten_cnt", "daycare_cnt", "bus_stop_cnt",
    "cctv_cnt", "cctv_cam_cnt", "speedbump_cnt",
]

# 인프라 공백 산출에 사용할 안전시설 6종
# 교육시설(school, kindergarten, daycare)은 위험 유발 시설로
# "없어도 공백이 아님" → 공백 지표에서 제외
SAFETY_COLS = [
    "crosswalk_cnt",   # 횡단보도    — 보행 안전 직결
    "child_zone_cnt",  # 어린이보호구역 — 어린이 안전 직결
    "speedbump_cnt",   # 과속방지턱  — 속도 억제
    "cctv_cnt",        # CCTV 개소  — 감시·억제
    "cctv_cam_cnt",    # CCTV 대수  — 감시·억제 보완
    "bus_stop_cnt",    # 버스정류장  — 교통약자 집산지
]
MAX_GAP = len(SAFETY_COLS)   # 6

HIGH_RISK_PERCENTILE = 75   # 상위 25%를 고위험으로 정의


# ── 통계 유틸 ────────────────────────────────────────────────────────────────

def _mean(values):
    return sum(values) / len(values) if values else 0.0


def pearson(x, y):
    n = len(x)
    if n < 2:
        return None
    mx, my = _mean(x), _mean(y)
    num = sum((xi - mx) * (yi - my) for xi, yi in zip(x, y))
    dx  = math.sqrt(sum((xi - mx) ** 2 for xi in x))
    dy  = math.sqrt(sum((yi - my) ** 2 for yi in y))
    if dx == 0 or dy == 0:
        return None
    return round(num / (dx * dy), 4)


def rank_list(values):
    """평균 순위(동점 처리 포함) 반환"""
    indexed = sorted(enumerate(values), key=lambda t: t[1])
    ranks   = [0.0] * len(values)
    i = 0
    while i < len(indexed):
        j = i
        while j < len(indexed) - 1 and indexed[j + 1][1] == indexed[j][1]:
            j += 1
        avg_rank = (i + j) / 2 + 1
        for k in range(i, j + 1):
            ranks[indexed[k][0]] = avg_rank
        i = j + 1
    return ranks


def spearman(x, y):
    return pearson(rank_list(x), rank_list(y))


def percentile(values, p):
    sv  = sorted(values)
    idx = (p / 100) * (len(sv) - 1)
    lo  = int(idx)
    hi  = min(lo + 1, len(sv) - 1)
    return sv[lo] + (idx - lo) * (sv[hi] - sv[lo])


# ── 메인 ─────────────────────────────────────────────────────────────────────

In [ ]:
print("=" * 60)
print("STEP 06 - 인프라 공백 분석")
print("=" * 60)

# 1. 데이터 로드
rows = []
with open(INPUT_PATH, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        rows.append(row)
print(f"\n[1] 격자 수: {len(rows):,}개 로드 완료")

epdo_vals = [float(r["epdo_total"]) for r in rows]

## ── 분석 1: 상관관계

In [ ]:
print("\n[2] EPDO × 시설물 상관관계 분석 중...")
corr_result = []
for col in INFRA_COLS:
    infra_vals    = [float(r.get(col, 0) or 0) for r in rows]
    pr            = pearson(epdo_vals, infra_vals)
    sr            = spearman(epdo_vals, infra_vals)
    infra_nonzero = sum(1 for v in infra_vals if v > 0)
    infra_mean    = _mean(infra_vals)

    if   (pr or 0) >  0.1: interp = "EPDO 높을수록 시설 많음 (노출 효과)"
    elif (pr or 0) < -0.1: interp = "EPDO 높을수록 시설 적음 (억제 효과 가능)"
    else:                   interp = "상관 미미"

    corr_result.append({
        "infra_col":         col,
        "pearson_r":         pr,
        "spearman_r":        sr,
        "infra_nonzero_cnt": infra_nonzero,
        "infra_mean":        round(infra_mean, 4),
        "해석":              interp,
    })

corr_result.sort(key=lambda x: abs(x["pearson_r"] or 0), reverse=True)

os.makedirs(os.path.dirname(CORR_OUTPUT), exist_ok=True)
with open(CORR_OUTPUT, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(corr_result[0].keys()))
    w.writeheader()
    w.writerows(corr_result)

print(f"\n  {'시설물':20s} {'Pearson':>9} {'Spearman':>10}  해석")
print("  " + "-" * 65)
for r in corr_result:
    print(f"  {r['infra_col']:20s} {str(r['pearson_r']):>9} {str(r['spearman_r']):>10}  {r['해석']}")

## ── 분석 2: 고위험 격자 인프라 공백

In [ ]:
print(f"\n[3] 고위험 격자 인프라 공백 추출 (상위 {100 - HIGH_RISK_PERCENTILE}%)...")
threshold = percentile(epdo_vals, HIGH_RISK_PERCENTILE)
print(f"    epdo_total 기준값: {threshold:.1f}점")

infra_means = {
    col: _mean([float(r.get(col, 0) or 0) for r in rows])
    for col in INFRA_COLS
}

gap_result = []
for r in rows:
    epdo = float(r["epdo_total"])
    if epdo < threshold:
        continue

    # 공백 산출: SAFETY_COLS 6종만 (교육시설 제외)
    gaps = []
    for col in SAFETY_COLS:
        val = float(r.get(col, 0) or 0)
        if val == 0:
            gaps.append(f"{col}(없음)")
        elif val < infra_means[col] * 0.5:
            gaps.append(f"{col}(부족)")

    gap_result.append({
        "grid_gid":          r["grid_gid"],
        "epdo_total":        epdo,
        "accident_cnt":      int(r["accident_cnt"]),
        "사망_cnt":           int(r["사망_cnt"]),
        "중상_cnt":           int(r["중상_cnt"]),
        "crosswalk_cnt":     int(float(r["crosswalk_cnt"])),
        "cctv_cnt":          int(float(r["cctv_cnt"])),
        "speedbump_cnt":     int(float(r["speedbump_cnt"])),
        "child_zone_cnt":    int(float(r["child_zone_cnt"])),
        # 학교인근 라벨 판단용 (교육시설은 공백 지표에서 제외했으나 라벨 부여에 활용)
        "school_cnt":        int(float(r.get("school_cnt", 0) or 0)),
        "kindergarten_cnt":  int(float(r.get("kindergarten_cnt", 0) or 0)),
        "gap_cnt":           len(gaps),
        "gap_items":         " | ".join(gaps) if gaps else "-",
    })

gap_result.sort(key=lambda x: (-x["gap_cnt"], -x["epdo_total"]))

with open(GAP_OUTPUT, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(gap_result[0].keys()))
    w.writeheader()
    w.writerows(gap_result)

total_high  = len(gap_result)
many_gaps   = sum(1 for r in gap_result if r["gap_cnt"] >= MAX_GAP // 2)
print(f"    고위험 격자: {total_high:,}개")
print(f"    시설물 절반 이상 공백: {many_gaps:,}개 ({many_gaps/total_high*100:.1f}%)")

print(f"\n    공백 심각 상위 10개:")
print(f"  {'격자ID':12s} {'EPDO':>7} {'사고':>4} {'공백수':>5}  공백 항목")
print("  " + "-" * 75)
for r in gap_result[:10]:
    items = r["gap_items"][:50] + ("..." if len(r["gap_items"]) > 50 else "")
    print(f"  {r['grid_gid']:12s} {r['epdo_total']:>7.0f} {r['accident_cnt']:>4}건 {r['gap_cnt']:>5}개  {items}")

print(f"\n저장: {CORR_OUTPUT}")
print(f"저장: {GAP_OUTPUT}")
print("=" * 60)

---

## 07_LINK_SPEED_CONGESTION

# STEP 07 - 링크 위험도 보강: 평균속도 + 혼잡강도

- 입력: epdo_analysis/output/03_링크별_위험도.csv
        data/09._평균속도.csv          (v_link_id, timeslot, velocity_AVRG)
        data/11._혼잡빈도강도.csv       (v_link_id, FRIN_CG)
        data/12._혼잡시간강도.csv       (v_link_id, TI_CG)
        data/08.상세도로망_네트워크.geojson (v_link_id → link_id 매핑)
- 출력: epdo_analysis/output/07_링크_속도혼잡_보강.csv

속도 보정 위험도:
  speed_adjusted_rate = epdo_rate × (avg_speed / 60.0)
  - 60km/h 기준: 초과할수록 보행자 치사율 급증 (교통안전공단 기준)
  - avg_speed < 60이면 가중치 < 1 (상대적으로 낮게 보정)

혼잡 위험도:
  congestion_risk = FRIN_CG × TI_CG / 100
  - FRIN: 혼잡빈도강도(얼마나 자주 막히는가, 0~100)
  - TI  : 혼잡시간강도(막혔을 때 얼마나 심한가, 0~100)

종합 보정 위험도:
  enhanced_rate = epdo_rate × speed_weight × (1 + congestion_weight)

In [ ]:
import csv
import json
import os
from collections import defaultdict

RISK_FILE     = os.path.join(EPDO_DIR, "epdo_analysis", "output", "03_링크별_위험도.csv")
SPEED_FILE    = os.path.join(BASE_DIR, "data", "09._평균속도.csv")
FRIN_FILE     = os.path.join(BASE_DIR, "data", "11._혼잡빈도강도.csv")
TI_FILE       = os.path.join(BASE_DIR, "data", "12._혼잡시간강도.csv")
ROAD_FILE     = os.path.join(BASE_DIR, "data", "08.상세도로망_네트워크.geojson")
OUTPUT_PATH   = os.path.join(EPDO_DIR, "epdo_analysis", "output", "07_링크_속도혼잡_보강.csv")

SPEED_BASE    = 60.0   # km/h 기준 (보행자 치사율 급증 기준)


def build_vlink_to_link(road_file):
    """v_link_id(상행/하행) → link_id 매핑 딕셔너리 생성 (STEP 02와 동일 로직)"""
    with open(road_file, "r", encoding="utf-8") as f:
        road = json.load(f)
    mapping = {}
    for feat in road["features"]:
        p   = feat["properties"]
        lid = str(p["link_id"])
        for key in ("up_v_link", "dw_v_link"):
            v = p.get(key)
            if v:
                mapping[str(v)] = lid
    return mapping

print("=" * 60)
print("STEP 07 - 링크 위험도 보강: 속도 + 혼잡강도")
print("=" * 60)

# 1. v_link_id → link_id 매핑
print("\n[1] 도로망 매핑 로드 중...")
vlink_to_link = build_vlink_to_link(ROAD_FILE)
print(f"    v_link → link 매핑 수: {len(vlink_to_link):,}")

# 2. 평균속도 집계 (timeslot별 전체 평균, 전 시간대 합산)
print("\n[2] 평균속도 로드 중...")
speed_sum   = defaultdict(float)
speed_cnt   = defaultdict(int)
peak_speed  = defaultdict(list)   # 07~09시(출근), 16~19시(하교/퇴근)

with open(SPEED_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        vid  = row["v_link_id"]
        lid  = vlink_to_link.get(vid)
        if not lid:
            continue
        try:
            slot = int(row["timeslot"])
            spd  = float(row["velocity_AVRG"])
        except (ValueError, KeyError):
            continue
        speed_sum[lid] += spd
        speed_cnt[lid] += 1
        if slot in (7, 8, 9, 16, 17, 18, 19):   # 교통약자 위험 시간대
            peak_speed[lid].append(spd)

link_avg_speed  = {
    lid: round(speed_sum[lid] / speed_cnt[lid], 2)
    for lid in speed_sum
}
link_peak_speed = {
    lid: round(sum(v) / len(v), 2)
    for lid, v in peak_speed.items() if v
}
print(f"    속도 매칭 링크: {len(link_avg_speed):,}개")

# 3. 혼잡빈도강도 (FRIN_CG)
print("\n[3] 혼잡빈도강도 로드 중...")
frin_by_link = defaultdict(list)
with open(FRIN_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        vid = row["v_link_id"]
        lid = vlink_to_link.get(vid)
        if lid:
            try:
                frin_by_link[lid].append(float(row["FRIN_CG"]))
            except ValueError:
                pass
link_frin = {lid: round(sum(v)/len(v), 4) for lid, v in frin_by_link.items()}
print(f"    혼잡빈도 매칭 링크: {len(link_frin):,}개")

# 4. 혼잡시간강도 (TI_CG)
print("\n[4] 혼잡시간강도 로드 중...")
ti_by_link = defaultdict(list)
with open(TI_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        vid = row["v_link_id"]
        lid = vlink_to_link.get(vid)
        if lid:
            try:
                ti_by_link[lid].append(float(row["TI_CG"]))
            except ValueError:
                pass
link_ti = {lid: round(sum(v)/len(v), 4) for lid, v in ti_by_link.items()}
print(f"    혼잡시간 매칭 링크: {len(link_ti):,}개")

# 5. 기존 링크 위험도 로드
print("\n[5] 기존 링크 위험도 로드 중...")
risk_rows = []
with open(RISK_FILE, "r", encoding="utf-8-sig") as f:
    risk_rows = list(csv.DictReader(f))
print(f"    링크 수: {len(risk_rows):,}개")

# 6. 보강 계산
print("\n[6] 속도·혼잡 보강 위험도 산출 중...")
result = []
matched_speed = matched_frin = matched_ti = 0

for r in risk_rows:
    lid        = r["link_id"]
    epdo_rate  = r["epdo_rate"]

    avg_spd    = link_avg_speed.get(lid)
    peak_spd   = link_peak_speed.get(lid)
    frin       = link_frin.get(lid)
    ti         = link_ti.get(lid)

    if avg_spd is not None:
        matched_speed += 1
    if frin is not None:
        matched_frin += 1
    if ti is not None:
        matched_ti += 1

    # 속도 가중치: avg_speed / 60 (60km/h 미만이면 1 미만)
    speed_weight = round(avg_spd / SPEED_BASE, 4) if avg_spd is not None else None

    # 혼잡 위험도: FRIN × TI / 100 (두 값 모두 있을 때만)
    congestion_risk = None
    if frin is not None and ti is not None:
        congestion_risk = round(frin * ti / 100, 4)

    # 종합 보정 위험도 (epdo_rate × 속도 × (1 + 혼잡))
    enhanced_rate = None
    if epdo_rate and epdo_rate != "" and speed_weight is not None:
        base = float(epdo_rate)
        cong_factor = (1 + congestion_risk / 100) if congestion_risk else 1.0
        enhanced_rate = round(base * speed_weight * cong_factor, 4)

    # 분류 레이블
    if avg_spd is not None:
        if avg_spd >= 80:
            speed_label = "고속(80+)"
        elif avg_spd >= 60:
            speed_label = "준고속(60~80)"
        elif avg_spd >= 40:
            speed_label = "중속(40~60)"
        else:
            speed_label = "저속(40미만)"
    else:
        speed_label = ""

    result.append({
        "link_id":          lid,
        "road_name":        r["road_name"],
        "road_rank":        r["road_rank"],
        "accident_cnt":     r["accident_cnt"],
        "epdo_total":       r["epdo_total"],
        "epdo_rate":        epdo_rate,
        "epdo_rank":        r["epdo_rank"],
        # 속도
        "avg_speed":        avg_spd if avg_spd is not None else "",
        "peak_speed":       peak_spd if peak_spd is not None else "",
        "speed_label":      speed_label,
        "speed_weight":     speed_weight if speed_weight is not None else "",
        # 혼잡
        "frin_cg":          frin if frin is not None else "",
        "ti_cg":            ti if ti is not None else "",
        "congestion_risk":  congestion_risk if congestion_risk is not None else "",
        # 보정 위험도
        "enhanced_rate":    enhanced_rate if enhanced_rate is not None else "",
    })

# 보정 위험도 기준 재정렬 + 순위 부여
result.sort(key=lambda x: (x["enhanced_rate"] == "", -(x["enhanced_rate"] or 0)))
rank = 1
for r in result:
    if r["enhanced_rate"] != "":
        r["enhanced_rank"] = rank
        rank += 1
    else:
        r["enhanced_rank"] = ""

# 7. 저장
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
cols = list(result[0].keys()) + ["enhanced_rank"]
cols = list(dict.fromkeys(cols))   # 중복 제거
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(result)

print(f"\n[결과]")
print(f"    전체 링크: {len(result):,}개")
print(f"    속도 매칭: {matched_speed:,}개 ({matched_speed/len(result)*100:.1f}%)")
print(f"    혼잡빈도 매칭: {matched_frin:,}개 ({matched_frin/len(result)*100:.1f}%)")
print(f"    혼잡시간 매칭: {matched_ti:,}개 ({matched_ti/len(result)*100:.1f}%)")

enh = [r for r in result if r["enhanced_rate"] != ""]
print(f"    종합보정 위험도 산출: {len(enh):,}개")
print(f"\n    보정 위험도 상위 10개:")
print(f"  {'순위':>4} {'도로명':15s} {'원래rate':>12} {'속도':>6} {'혼잡':>6} {'보정rate':>14}")
print("  " + "-" * 70)
for r in result[:10]:
    if r["enhanced_rate"] == "":
        continue
    print(f"  {r['enhanced_rank']:>4} {str(r['road_name']):15s} "
          f"{float(r['epdo_rate']):>12.1f} "
          f"{str(r['avg_speed']):>6} "
          f"{str(r['congestion_risk']):>6} "
          f"{float(r['enhanced_rate']):>14.1f}")

print(f"\n저장: {OUTPUT_PATH}")
print("=" * 60)

---

## 07_ML_RISK_MODEL

# STEP 07 - ML 기반 교통사고 위험도 예측 (Random Forest)

- 입력: output/격자별_통합데이터.csv          (17,577개 전체 격자 인프라)
        epdo_analysis/output/05_격자별_EPDO_인프라통합.csv (5,723개 사고 격자 EPDO)
- 출력: epdo_analysis/output/07_ML_피처중요도.csv
        epdo_analysis/output/07_ML_위험도예측.csv

모델 설명:
  - 전체 17,577개 격자를 대상으로 학습 (사고 없는 격자 → 저위험)
  - 고위험 기준: epdo_total ≥ 69점 (전체 사고 격자 상위 25%)
  - 인프라 변수 12개를 Feature로 사용
  - class_weight='balanced': 저위험(11,854개) vs 고위험(1,480개) 불균형 보정
  - 활용: 하남교산 등 신도시의 인프라 계획 데이터만으로 사전 위험도 예측 가능

In [ ]:
import csv
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, roc_auc_score)
from sklearn.model_selection import train_test_split

GRID_FILE   = os.path.join(EPDO_DIR, "epdo_analysis", "output", "00_격자별_통합데이터.csv")
EPDO_FILE   = os.path.join(EPDO_DIR, "epdo_analysis", "output", "05_격자별_EPDO_인프라통합.csv")
OUT_FEAT    = os.path.join(EPDO_DIR, "epdo_analysis", "output", "07_ML_피처중요도.csv")
OUT_PRED    = os.path.join(EPDO_DIR, "epdo_analysis", "output", "07_ML_위험도예측.csv")

HIGH_RISK_THRESHOLD = 69.0   # epdo_total 기준 고위험 (상위 25%)

FEATURES = [
    "crosswalk_cnt", "child_zone_cnt", "school_cnt",
    "kindergarten_cnt", "kindergarten_child_cnt", "daycare_cnt",
    "bus_stop_cnt", "cctv_cnt", "cctv_cam_cnt",
    "speedbump_cnt", "speedbump_hght_below", "speedbump_hght_above",
]

In [ ]:
print("=" * 60)
print("STEP 07 - ML 기반 교통사고 위험도 예측 (Random Forest)")
print("=" * 60)

## ── 1. 데이터 로드

In [ ]:
print("\n[1] 데이터 로드 중...")
grid_df = pd.read_csv(GRID_FILE, encoding="utf-8-sig")
epdo_df = pd.read_csv(EPDO_FILE, encoding="utf-8-sig")

print(f"    전체 격자: {len(grid_df):,}개")
print(f"    사고 격자: {len(epdo_df):,}개")

## ── 2. 데이터 병합

In [ ]:
print("\n[2] 데이터 병합 중...")

# EPDO에서 필요한 컬럼만 추출
epdo_slim = epdo_df[["grid_gid", "epdo_total"]].copy()

# LEFT JOIN: 전체 격자 + EPDO (없으면 0)
df = grid_df.merge(epdo_slim, on="grid_gid", how="left")
df["epdo_total"] = df["epdo_total"].fillna(0.0)

# 고위험 라벨 생성
df["is_high_risk"] = (df["epdo_total"] >= HIGH_RISK_THRESHOLD).astype(int)

high = df["is_high_risk"].sum()
low  = len(df) - high
print(f"    고위험(1): {high:,}개 ({high/len(df)*100:.1f}%)")
print(f"    저위험(0): {low:,}개 ({low/len(df)*100:.1f}%)")

## ── 3. Feature / Target 구성

In [ ]:
print("\n[3] 피처 구성 중...")
for col in FEATURES:
    if col not in df.columns:
        df[col] = 0
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

X = df[FEATURES].values
y = df["is_high_risk"].values

## ── 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.2, random_state=42, stratify=y
)
print(f"    훈련셋: {len(X_train):,}개 | 테스트셋: {len(X_test):,}개")

## ── 5. 모델 학습

In [ ]:
print("\n[4] Random Forest 학습 중...")
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",   # 불균형 보정
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
print("    학습 완료")

## ── 6. 평가

In [ ]:
print("\n[5] 모델 평가...")
y_pred      = model.predict(X_test)
y_prob      = model.predict_proba(X_test)[:, 1]
auc         = roc_auc_score(y_test, y_prob)

print(f"\n  ROC-AUC:  {auc:.4f}")
print(f"\n  분류 리포트:")
print(classification_report(y_test, y_pred,
                             target_names=["저위험", "고위험"],
                             digits=3))

cm = confusion_matrix(y_test, y_pred)
print(f"  Confusion Matrix:")
print(f"            예측저위험  예측고위험")
print(f"  실제저위험  {cm[0,0]:>7}   {cm[0,1]:>7}")
print(f"  실제고위험  {cm[1,0]:>7}   {cm[1,1]:>7}")

## ── 7. 피처 중요도 저장

In [ ]:
print("\n[6] 피처 중요도 저장 중...")
importances = model.feature_importances_
feat_rows = sorted(
    [{"feature": f, "importance": round(float(imp), 6)}
     for f, imp in zip(FEATURES, importances)],
    key=lambda x: -x["importance"]
)

os.makedirs(os.path.dirname(OUT_FEAT), exist_ok=True)
with open(OUT_FEAT, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["rank", "feature", "importance"])
    w.writeheader()
    for i, row in enumerate(feat_rows, 1):
        w.writerow({"rank": i, **row})

print(f"\n  {'순위':>4} {'피처':25s} {'중요도':>10}")
print("  " + "-" * 44)
for i, r in enumerate(feat_rows, 1):
    bar = "#" * int(r["importance"] * 200)
    print(f"  {i:>4} {r['feature']:25s} {r['importance']:>10.4f}  {bar}")

## ── 8. 전체 격자 예측 결과 저장

In [ ]:
print("\n[7] 전체 격자 예측 중...")
all_prob  = model.predict_proba(X)[:, 1]
all_pred  = model.predict(X)

df["risk_prob"]     = np.round(all_prob, 4)
df["predicted_risk"]= all_pred

out_cols = ["grid_gid", "epdo_total", "is_high_risk", "risk_prob", "predicted_risk"]
df[out_cols].to_csv(OUT_PRED, index=False, encoding="utf-8-sig")

# 요약
pred_high = int(all_pred.sum())
print(f"    예측 고위험 격자: {pred_high:,}개 (실제: {high:,}개)")

print(f"\n  risk_prob 상위 10개 격자 (예측 위험도 높은 순):")
top10 = df.nlargest(10, "risk_prob")[
    ["grid_gid", "epdo_total", "is_high_risk", "risk_prob"]
]
print(f"  {'격자ID':12s} {'실제EPDO':>9} {'실제라벨':>6} {'위험확률':>8}")
print("  " + "-" * 42)
for _, r in top10.iterrows():
    label = "고위험" if r["is_high_risk"] else "저위험"
    print(f"  {r['grid_gid']:12s} {r['epdo_total']:>9.1f}  {label:>6}  {r['risk_prob']:>8.4f}")

print(f"\n저장: {OUT_FEAT}")
print(f"저장: {OUT_PRED}")
print("=" * 60)

---

## 08_GRID_COMPOSITE_RISK

# STEP 08 - 격자 종합 위험 지수 산출 (v2 - 전체 데이터 통합)

- 입력:
    epdo_analysis/output/06_인프라공백_고위험격자.csv  (기준: 고위험 격자)
    epdo_analysis/output/07_링크_속도혼잡_보강.csv     (링크별 속도·혼잡 ← STEP 07)
    data/01._격자_(4개_시·구).geojson                  (격자 폴리곤)
    data/03._성연령별_거주인구(격자).csv                (gid → 노인 거주 인구)
    data/04._성연령별_유동인구.csv                      (lon/lat → 어린이·노인 유동인구)
    data/05._시간대별_직장인구.csv                      (lon/lat → 시간대별 직장인구)
    data/06._시간대별_방문인구.csv                      (lon/lat → 시간대별 방문인구)
    data/07._주중주말_서비스인구.csv                    (lon/lat → 주중/주말 인구)
    data/08.상세도로망_네트워크.geojson                 (링크 중심 → 격자 속도 집계용)
    data/22._토지이용계획도_(4개_신도시).geojson        (폴리곤 → 토지이용 유형)
    data/23._토지이용계획도_(하남교산).geojson          (하남교산 예측 위험도 출력용)
- 출력:
    epdo_analysis/output/08_격자_종합위험지수.csv       (고위험 격자 종합 분석)
    epdo_analysis/output/08_하남교산_예측위험도.csv     (하남교산 신도시 예측)

수정 내역 (v2):
  1. CRS 경고 수정: centroid 계산 전 EPSG:5186(한국 중부원점) 재투영
  2. STEP 07 속도·혼잡 → 격자 단위 통합
     (도로망 링크 중심 → 격자 공간 join → 격자별 평균 속도·혼잡 집계)
  3. 토지이용 매칭률 개선: within → intersects (격자 폴리곤 기준)
  4. 23번 하남교산 예측 위험도 추가 출력
  5. "통행피크위험" 라벨 기준 개선: 중앙값 기준 상위 50%만 라벨 부여

종합 위험 지수 산출 공식:
  composite_risk = epdo_total
                   × vuln_correction   (교통약자 보정: 1 + 노인거주비율 + 어린이유동비율)
                   × infra_penalty     (인프라 공백: 1 + gap_cnt/9)
                   × speed_weight      (속도 보정: avg_speed/60, 없으면 1.0)

In [ ]:
import csv
import json
import os
import warnings
from collections import defaultdict, Counter

import geopandas as gpd
from shapely.geometry import Point

warnings.filterwarnings("ignore", category=UserWarning)   # CRS 관련 경고 억제

# 입력
GAP_FILE     = os.path.join(EPDO_DIR, "epdo_analysis", "output", "06_인프라공백_고위험격자.csv")
STEP07_FILE  = os.path.join(EPDO_DIR, "epdo_analysis", "output", "07_링크_속도혼잡_보강.csv")
GRID_FILE    = os.path.join(BASE_DIR, "data", "01._격자_(4개_시·구).geojson")
RES_FILE     = os.path.join(BASE_DIR, "data", "03._성연령별_거주인구(격자).csv")
FLOAT_FILE   = os.path.join(BASE_DIR, "data", "04._성연령별_유동인구.csv")
WORK_FILE    = os.path.join(BASE_DIR, "data", "05._시간대별_직장인구.csv")
VISIT_FILE   = os.path.join(BASE_DIR, "data", "06._시간대별_방문인구.csv")
SVC_FILE     = os.path.join(BASE_DIR, "data", "07._주중주말_서비스인구.csv")
ROAD_FILE    = os.path.join(BASE_DIR, "data", "08.상세도로망_네트워크.geojson")
LU_FILE      = os.path.join(BASE_DIR, "data", "22._토지이용계획도_(4개_신도시).geojson")
LU_HANAM     = os.path.join(BASE_DIR, "data", "23._토지이용계획도_(하남교산).geojson")

# 출력
OUTPUT_PATH  = os.path.join(EPDO_DIR, "epdo_analysis", "output", "08_격자_종합위험지수.csv")
HANAM_OUTPUT = os.path.join(EPDO_DIR, "epdo_analysis", "output", "08_하남교산_예측위험도.csv")

CRS_PROJ     = "EPSG:5186"   # 한국 중부원점 (단위: 미터, centroid 계산 정확)
CRS_GEO      = "EPSG:4326"   # WGS84

# 교통약자 취약 시간대: 등교(07~09), 하교(13~16), 퇴근(17~19)
VULN_SLOTS   = {7, 8, 9, 13, 14, 15, 16, 17, 18, 19}
SLOT_COLS    = [f"TMST_{str(h).zfill(2)}" for h in VULN_SLOTS]
ALL_SLOTS    = [f"TMST_{str(h).zfill(2)}" for h in range(24)]

# 토지이용 → 위험 카테고리 매핑
SCHOOL_TYPES    = {"학교", "초등학교", "중학교", "고등학교", "교육시설"}
RESIDENT_TYPES  = {"아파트", "연립주택", "다세대주택", "단독주택", "공동주택", "주상복합", "연립"}
COMMERCIAL_TYPES = {"상업", "일반상업", "근린상업", "근린생활시설용지", "상업시설"}


# ── 유틸 ────────────────────────────────────────────────────────────────────

def load_csv_dict(path, key_col):
    with open(path, "r", encoding="utf-8-sig") as f:
        return {row[key_col]: row for row in csv.DictReader(f)}


def safe_float(val, default=0.0):
    try:
        return float(val or 0)
    except (ValueError, TypeError):
        return default


def points_to_grid(lon_col, lat_col, rows, value_cols, grid_gdf):
    """lon/lat 포인트 → 격자 공간 join 후 격자별 집계 반환"""
    pts = []
    for row in rows:
        try:
            lon, lat = float(row[lon_col]), float(row[lat_col])
        except (ValueError, KeyError):
            continue
        rec = {"geometry": Point(lon, lat)}
        for c in value_cols:
            rec[c] = safe_float(row.get(c))
        pts.append(rec)

    pts_gdf  = gpd.GeoDataFrame(pts, crs=CRS_GEO)
    joined   = gpd.sjoin(pts_gdf, grid_gdf[["gid", "geometry"]], how="left", predicate="within")
    return joined


def median(values):
    sv = sorted(v for v in values if v is not None)
    if not sv:
        return 0
    mid = len(sv) // 2
    return (sv[mid - 1] + sv[mid]) / 2 if len(sv) % 2 == 0 else sv[mid]


# ── 메인 ─────────────────────────────────────────────────────────────────────

In [ ]:
print("=" * 60)
print("STEP 08 - 격자 종합 위험 지수 산출 (v2)")
print("=" * 60)

## ── 1. 고위험 격자 목록

In [ ]:
print("\n[1] 고위험 격자 로드 중...")
gap_data = load_csv_dict(GAP_FILE, "grid_gid")
print(f"    고위험 격자 수: {len(gap_data):,}개")

## ── 2. 격자 GeoDataFrame (고위험만 필터)

In [ ]:
print("\n[2] 격자 폴리곤 로드 중...")
grid_gdf = gpd.read_file(GRID_FILE)
grid_gdf = grid_gdf[grid_gdf["gid"].isin(gap_data.keys())].reset_index(drop=True)
print(f"    고위험 격자 폴리곤: {len(grid_gdf):,}개")

## ── 3. STEP 07 속도·혼잡 → 격자 단위 집계

In [ ]:
# 개선 v3:
#   [문제1] 중심점 기반 → 링크 선분 intersects 기반으로 변경
#           (중심점이 다른 격자에 떨어지는 오류 해소)
#   [문제2] 속도 미계측 링크 → 도로 등급별 평균으로 보간
print("\n[3] STEP 07 속도·혼잡 → 격자 단위 집계 중...")
step07 = load_csv_dict(STEP07_FILE, "link_id")

# 도로망 로드 (speed 데이터 있는 링크만)
road_gdf_all = gpd.read_file(ROAD_FILE)
road_gdf_all["link_id"] = road_gdf_all["link_id"].astype(str)

road_gdf_spd = road_gdf_all[road_gdf_all["link_id"].isin(step07.keys())].copy()

# 속도·혼잡 컬럼 추가
road_gdf_spd["avg_speed"]       = road_gdf_spd["link_id"].map(
    lambda lid: safe_float(step07.get(lid, {}).get("avg_speed")))
road_gdf_spd["congestion_risk"] = road_gdf_spd["link_id"].map(
    lambda lid: safe_float(step07.get(lid, {}).get("congestion_risk")))

# [문제1 해결] centroid 제거 → 선분 geometry 그대로 intersects 조인
road_proj = road_gdf_spd.to_crs(CRS_PROJ)
grid_proj = grid_gdf.to_crs(CRS_PROJ)

link_grid = gpd.sjoin(
    road_proj[["link_id", "avg_speed", "congestion_risk", "geometry"]],
    grid_proj[["gid", "geometry"]],
    how="inner",
    predicate="intersects"   # 선분이 격자와 교차하면 모두 매칭
)
speed_agg = {}
for gid, grp in link_grid.groupby("gid"):
    spd_vals = [v for v in grp["avg_speed"] if v > 0]
    cng_vals = [v for v in grp["congestion_risk"] if v > 0]
    speed_agg[str(gid)] = {
        "grid_avg_speed":       round(sum(spd_vals) / len(spd_vals), 2) if spd_vals else None,
        "grid_congestion_risk": round(sum(cng_vals) / len(cng_vals), 4) if cng_vals else None,
    }
matched_before_interp = len(speed_agg)
print(f"    [문제1 해결 후] 속도·혼잡 매칭 격자: {matched_before_interp:,}개 "
      f"({matched_before_interp/len(gap_data)*100:.1f}%)")

# [문제2 해결] 속도 미계측 링크 → 도로 등급별 평균 보간
# 1단계: 전체 도로망에서 road_rank별 평균 속도 계산 (계측 링크 기준)
rank_speed_map = defaultdict(list)
rank_cong_map  = defaultdict(list)
for lid, data in step07.items():
    spd = safe_float(data.get("avg_speed"))
    cng = safe_float(data.get("congestion_risk"))
    # 해당 link_id의 road_rank 조회
    rank_rows = road_gdf_all[road_gdf_all["link_id"] == lid]
    if rank_rows.empty:
        continue
    rank = str(rank_rows.iloc[0].get("road_rank", ""))
    if spd > 0 and rank:
        rank_speed_map[rank].append(spd)
    if cng > 0 and rank:
        rank_cong_map[rank].append(cng)

rank_avg_speed = {r: round(sum(v)/len(v), 2) for r, v in rank_speed_map.items()}
rank_avg_cong  = {r: round(sum(v)/len(v), 4) for r, v in rank_cong_map.items()}
overall_avg_speed = round(sum(rank_avg_speed.values())/len(rank_avg_speed), 2) \
                    if rank_avg_speed else 30.0
overall_avg_cong  = round(sum(rank_avg_cong.values())/len(rank_avg_cong), 4) \
                    if rank_avg_cong else 0.0

# 2단계: 결측 격자에 해당 격자 내 링크의 도로 등급별 평균 적용
# 전체 도로망(속도 없는 링크 포함)과 격자 intersects 조인
road_all_proj = road_gdf_all.to_crs(CRS_PROJ)
link_grid_all = gpd.sjoin(
    road_all_proj[["link_id", "road_rank", "geometry"]],
    grid_proj[["gid", "geometry"]],
    how="inner",
    predicate="intersects"
)
interp_cnt = 0
for gid, grp in link_grid_all.groupby("gid"):
    gid_str = str(gid)
    if gid_str in speed_agg:
        continue   # 이미 속도 데이터 있음
    # 격자 내 링크들의 도로 등급 목록
    ranks = [str(r) for r in grp["road_rank"].dropna() if str(r)]
    if not ranks:
        continue
    spd_vals = [rank_avg_speed.get(r, overall_avg_speed) for r in ranks]
    cng_vals = [rank_avg_cong.get(r,  overall_avg_cong)  for r in ranks]
    speed_agg[gid_str] = {
        "grid_avg_speed":       round(sum(spd_vals)/len(spd_vals), 2),
        "grid_congestion_risk": round(sum(cng_vals)/len(cng_vals), 4),
        "interpolated": True,   # 보간 여부 표시
    }
    interp_cnt += 1

total_matched = len(speed_agg)
print(f"    [문제2 해결 후] 도로등급 보간 추가: {interp_cnt:,}개")
print(f"    최종 속도·혼잡 커버 격자: {total_matched:,}개 "
      f"({total_matched/len(gap_data)*100:.1f}%)")
print(f"    도로 등급별 기본 속도: { {k: v for k, v in sorted(rank_avg_speed.items())} }")

## ── 4. 03번 거주인구 — gid 직접 join

In [ ]:
print("\n[4] 거주인구(노인 60대+) 로드 중...")
res_data = {}
ELDERLY_COLS = ["m_60g_pop", "w_60g_pop", "m_70g_pop", "w_70g_pop",
                "m_80g_pop", "w_80g_pop", "m_90g_pop", "w_90g_pop",
                "m_100g_pop", "w_100g_pop"]
ALL_RES_COLS = ["m_20g_pop", "w_20g_pop", "m_30g_pop", "w_30g_pop",
                "m_40g_pop", "w_40g_pop", "m_50g_pop", "w_50g_pop"] + ELDERLY_COLS
with open(RES_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        gid = row.get("gid", "")
        if gid not in gap_data:
            continue
        elderly = sum(safe_float(row.get(c)) for c in ELDERLY_COLS)
        total   = sum(safe_float(row.get(c)) for c in ALL_RES_COLS)
        res_data[gid] = {"elderly_pop": elderly, "total_res_pop": total}
print(f"    거주인구 매칭 격자: {len(res_data):,}개 "
      f"(비어있는 격자는 거주인구 0으로 처리)")

## ── 5. 04번 유동인구 — 공간 join

In [ ]:
print("\n[5] 유동인구(어린이·노인) 공간 join 중...")
float_cols = ["m_10g_pop", "w_10g_pop", "m_60g_pop", "w_60g_pop",
              "m_20g_pop", "w_20g_pop", "m_30g_pop", "w_30g_pop",
              "m_40g_pop", "w_40g_pop", "m_50g_pop", "w_50g_pop"]
with open(FLOAT_FILE, "r", encoding="utf-8-sig") as f:
    float_rows = list(csv.DictReader(f))
float_joined = points_to_grid("lon", "lat", float_rows, float_cols, grid_gdf)

float_agg = {}
for gid, grp in float_joined.dropna(subset=["gid"]).groupby("gid"):
    child   = grp["m_10g_pop"].sum() + grp["w_10g_pop"].sum()
    elderly = grp["m_60g_pop"].sum() + grp["w_60g_pop"].sum()
    total   = sum(grp[c].sum() for c in float_cols)
    float_agg[str(gid)] = {
        "child_float":   round(child, 4),
        "elderly_float": round(elderly, 4),
        "total_float":   round(total, 4),
    }
print(f"    유동인구 매칭 격자: {len(float_agg):,}개")

## ── 6. 05번 직장인구 — 취약 시간대 공간 join

In [ ]:
print("\n[6] 직장인구(취약 시간대) 공간 join 중...")
with open(WORK_FILE, "r", encoding="utf-8-sig") as f:
    work_rows = list(csv.DictReader(f))
work_joined = points_to_grid("lon", "lat", work_rows, SLOT_COLS + ALL_SLOTS, grid_gdf)

work_agg = {}
for gid, grp in work_joined.dropna(subset=["gid"]).groupby("gid"):
    vuln  = sum(grp[c].sum() for c in SLOT_COLS if c in grp.columns)
    total = sum(grp[c].sum() for c in ALL_SLOTS if c in grp.columns)
    work_agg[str(gid)] = {
        "vuln_work":  round(vuln, 4),
        "total_work": round(total, 4),
    }
print(f"    직장인구 매칭 격자: {len(work_agg):,}개")

## ── 7. 06번 방문인구 — 취약 시간대 공간 join

In [ ]:
print("\n[7] 방문인구(취약 시간대) 공간 join 중...")
with open(VISIT_FILE, "r", encoding="utf-8-sig") as f:
    visit_rows = list(csv.DictReader(f))
visit_joined = points_to_grid("lon", "lat", visit_rows, SLOT_COLS + ALL_SLOTS, grid_gdf)

visit_agg = {}
for gid, grp in visit_joined.dropna(subset=["gid"]).groupby("gid"):
    vuln  = sum(grp[c].sum() for c in SLOT_COLS if c in grp.columns)
    total = sum(grp[c].sum() for c in ALL_SLOTS if c in grp.columns)
    visit_agg[str(gid)] = {
        "vuln_visit":  round(vuln, 4),
        "total_visit": round(total, 4),
    }
print(f"    방문인구 매칭 격자: {len(visit_agg):,}개")

## ── 8. 07번 서비스인구 — 주중/주말 분리 공간 join

In [ ]:
print("\n[8] 서비스인구(주중/주말) 공간 join 중...")
svc_h, svc_w = [], []
with open(SVC_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        (svc_h if row.get("hw") == "H" else svc_w).append(row)

def agg_svc(rows, prefix):
    joined = points_to_grid("lon", "lat", rows, ["w_pop", "v_pop"], grid_gdf)
    agg = {}
    for gid, grp in joined.dropna(subset=["gid"]).groupby("gid"):
        agg[str(gid)] = {
            f"{prefix}_work":  round(grp["w_pop"].sum(), 4),
            f"{prefix}_visit": round(grp["v_pop"].sum(), 4),
        }
    return agg

wd_agg = agg_svc(svc_h, "weekday")
we_agg = agg_svc(svc_w, "weekend")
print(f"    주중 매칭: {len(wd_agg):,}개 | 주말 매칭: {len(we_agg):,}개")

## ── 9. 22번 토지이용 — 수정 2: intersects로 매칭률 개선

In [ ]:
print("\n[9] 토지이용(4개 신도시) 공간 join 중...")
lu_gdf = gpd.read_file(LU_FILE)

# within → intersects: 격자 폴리곤과 토지이용 폴리곤이 조금이라도 겹치면 매칭
lu_joined = gpd.sjoin(
    grid_gdf[["gid", "geometry"]],
    lu_gdf[["blockType", "geometry"]],
    how="left",
    predicate="intersects"
)
lu_map = {}
for gid, grp in lu_joined.dropna(subset=["blockType"]).groupby("gid"):
    lu_map[str(gid)] = grp["blockType"].value_counts().idxmax()
print(f"    토지이용 매칭 격자: {len(lu_map):,}개 "
      f"({len(lu_map)/len(gap_data)*100:.1f}%)")

## ── 10. 종합 위험 지수 산출

In [ ]:
print("\n[10] 종합 위험 지수 산출 중...")

# 취약 시간대 인구 중앙값 계산 (통행피크 라벨 기준)
vuln_pops = []
for gid in gap_data:
    vw = work_agg.get(gid, {}).get("vuln_work", 0)
    vv = visit_agg.get(gid, {}).get("vuln_visit", 0)
    vuln_pops.append(vw + vv)
vuln_median = median(vuln_pops)

# 교통약자 유동 비율 75번째 퍼센타일 계산 (교통약자유동多 라벨 기준 — 상위 25%만)
# 고정 임계값 0.15는 99.2%가 해당되어 의미없으므로 분포 기반 동적 기준 적용
vuln_float_vals = []
for gid in gap_data:
    flt = float_agg.get(gid, {})
    cf = flt.get("child_float", 0)
    ef = flt.get("elderly_float", 0)
    tf = flt.get("total_float", 0)
    rt = round((cf + ef) / tf, 4) if tf > 0 else 0
    vuln_float_vals.append(rt)
vuln_float_vals_sorted = sorted(vuln_float_vals)
vuln_float_p75 = vuln_float_vals_sorted[int(len(vuln_float_vals_sorted) * 0.75)]
print(f"    교통약자유동 75%ile 임계값: {vuln_float_p75:.4f} (상위 25% 기준 라벨 부여)")

result = []
for gid, gap in gap_data.items():
    epdo      = float(gap["epdo_total"])
    acc_cnt   = int(gap["accident_cnt"])
    death     = int(gap["사망_cnt"])
    heavy     = int(gap["중상_cnt"])
    gap_cnt        = int(gap["gap_cnt"])
    gap_items      = gap["gap_items"]
    # 학교인근 라벨용 교육시설 수 (공백 지표 제외이지만 라벨 판단에 사용)
    edu_cnt        = int(gap.get("school_cnt", 0) or 0) + int(gap.get("kindergarten_cnt", 0) or 0)

    # 거주인구 (03번)
    res            = res_data.get(gid, {})
    elderly_res    = res.get("elderly_pop", 0)
    total_res      = res.get("total_res_pop", 0)
    elderly_res_rt = round(elderly_res / total_res, 4) if total_res > 0 else 0

    # 유동인구 (04번)
    flt            = float_agg.get(gid, {})
    child_float    = flt.get("child_float", 0)
    elderly_float  = flt.get("elderly_float", 0)
    total_float    = flt.get("total_float", 0)
    vuln_float_rt  = round((child_float + elderly_float) / total_float, 4) \
                     if total_float > 0 else 0

    # 직장·방문인구 취약 시간대 (05·06번)
    vuln_work  = work_agg.get(gid, {}).get("vuln_work", 0)
    vuln_visit = visit_agg.get(gid, {}).get("vuln_visit", 0)
    vuln_peak  = vuln_work + vuln_visit

    # 서비스인구 주말 집중도 (07번)
    wd_visit      = wd_agg.get(gid, {}).get("weekday_visit", 0)
    we_visit      = we_agg.get(gid, {}).get("weekend_visit", 0)
    total_svc     = wd_visit + we_visit
    weekend_ratio = round(we_visit / total_svc, 4) if total_svc > 0 else 0

    # 속도·혼잡 (STEP 07 격자 집계)
    spd           = speed_agg.get(gid, {})
    grid_speed    = spd.get("grid_avg_speed")       # None 가능
    grid_cong     = spd.get("grid_congestion_risk") # None 가능
    speed_weight  = round(grid_speed / 60.0, 4) if grid_speed else 1.0

    # 토지이용 (22번)
    land_use = lu_map.get(gid, "미분류")
# 1순위: 교통약자 보정 (03 거주인구 + 04 유동인구)
    vuln_ratio      = elderly_res_rt + vuln_float_rt
    vuln_correction = round(1.0 + vuln_ratio, 4)   # 최소 1.0

    # 인프라 공백 패널티 (6개 안전시설 기준, 분모 6으로 변경)
    # 교육시설(학교·유치원·어린이집)은 STEP 06에서 제외됨
    infra_penalty   = round(1.0 + gap_cnt / 6, 4)  # 최소 1.0, 최대 2.0

    # 2순위: 속도 보정 (09 평균속도)
    # speed_weight 이미 위에서 계산됨 (grid_speed / 60, 없으면 1.0)

    # 4순위: 혼잡강도 보정 (11 FRIN_CG × 12 TI_CG)
    # grid_congestion = FRIN_CG × TI_CG / 100 (범위 0~100)
    # 혼잡할수록 위험도 증가: 1.0(혼잡 없음) ~ 2.0(최대 혼잡)
    if grid_cong is not None:
        congestion_factor = round(1.0 + grid_cong / 100, 4)
    else:
        congestion_factor = 1.0

    # 5순위: 취약 시간대 인구 보정 (05 직장인구 + 06 방문인구)
    # vuln_peak / vuln_median 비율로 정규화
    # 중앙값의 2배 이상이면 최대 1.5배, 중앙값 이하면 1.0
    if vuln_median > 0 and vuln_peak > vuln_median:
        peak_ratio  = vuln_peak / vuln_median
        peak_factor = round(1.0 + min((peak_ratio - 1) * 0.3, 0.5), 4)
    else:
        peak_factor = 1.0

    # 5순위: 주말 방문인구 보정 (07 서비스인구)
    # weekend_ratio 0.5 초과분만큼 최대 25% 가중
    weekend_factor = round(1.0 + max(0.0, weekend_ratio - 0.5) * 0.5, 4)

    # 종합 위험 지수 (1~5순위 전체 반영)
    composite_risk  = round(
        epdo
        * vuln_correction    # 1순위: 교통약자 인구
        * infra_penalty      # 기존: 인프라 공백
        * speed_weight       # 2순위: 속도
        * congestion_factor  # 4순위: 혼잡강도
        * peak_factor        # 5순위: 취약시간 인구
        * weekend_factor,    # 5순위: 주말 방문
        2
    )
    labels = []
    if elderly_res_rt > 0.2:
        labels.append("노인밀집거주")
    if vuln_float_rt > vuln_float_p75:   # 상위 25%만 (동적 임계값 ~0.34)
        labels.append("교통약자유동多")
    if weekend_ratio > 0.55:
        labels.append("주말방문집중")
    if vuln_peak > vuln_median:           # 중앙값 초과만 라벨 부여
        labels.append("통행피크위험")
    if grid_speed and grid_speed >= 60:
        labels.append("고속도로위험")
    # 학교인근: land_use 기반(부지 내) + edu_cnt>0(격자 내 학교·유치원 존재) 모두 포함
    if land_use in SCHOOL_TYPES or edu_cnt > 0:
        labels.append("학교인근")
    if land_use in RESIDENT_TYPES:
        labels.append("주거밀집")
    if land_use in COMMERCIAL_TYPES:
        labels.append("상업지역")
    risk_label = " | ".join(labels) if labels else "일반"

    result.append({
        "grid_gid":          gid,
        "epdo_total":        epdo,
        "accident_cnt":      acc_cnt,
        "사망_cnt":           death,
        "중상_cnt":           heavy,
        # 교통약자 인구
        "elderly_res_pop":   round(elderly_res, 2),
        "elderly_res_ratio": elderly_res_rt,
        "child_float_pop":   round(child_float, 4),
        "elderly_float_pop": round(elderly_float, 4),
        "vuln_float_ratio":  vuln_float_rt,
        # 시간대 인구
        "vuln_time_work":    round(vuln_work, 4),
        "vuln_time_visit":   round(vuln_visit, 4),
        "vuln_peak_pop":     round(vuln_peak, 4),
        # 주말 집중도
        "weekend_ratio":     weekend_ratio,
        # 속도·혼잡 (격자 단위)
        "grid_avg_speed":    grid_speed if grid_speed is not None else "",
        "grid_congestion":   grid_cong  if grid_cong  is not None else "",
        # 인프라 공백
        "gap_cnt":           gap_cnt,
        "gap_items":         gap_items,
        # 토지이용
        "land_use":          land_use,
        # 보정계수 (1~5순위 전체)
        "vuln_correction":   vuln_correction,    # 1순위
        "infra_penalty":     infra_penalty,       # 기존
        "speed_weight":      speed_weight,        # 2순위 (grid_avg_speed / 60)
        "congestion_factor": congestion_factor,   # 4순위
        "peak_factor":       peak_factor,         # 5순위
        "weekend_factor":    weekend_factor,      # 5순위
        # 종합 위험 지수
        "composite_risk":    composite_risk,
        # 위험 라벨
        "risk_label":        risk_label,
    })

result.sort(key=lambda x: -x["composite_risk"])
for i, r in enumerate(result, 1):
    r["composite_rank"] = i

## ── 11. 엔트로피 가중치 적용 종합 위험 지수 (STEP 09 결과 반영)

In [ ]:
print("\n[11] 엔트로피 가중치 적용 중...")

# STEP 09 CSV에서 엔트로피 가중치 동적 로드 (하드코딩 제거)
ENTROPY_W_PATH = os.path.join(EPDO_DIR, "epdo_analysis", "output", "09_엔트로피_가중치.csv")
ENTROPY_W = {}
try:
    with open(ENTROPY_W_PATH, "r", encoding="utf-8-sig") as _f:
        for _row in csv.DictReader(_f):
            ENTROPY_W[_row["인자"]] = float(_row["가중치"])
    print(f"    엔트로피 가중치 로드: {ENTROPY_W_PATH}")
except FileNotFoundError:
    # 09번 미실행 시 fallback — 최근 계산값 사용
    print(f"    ⚠ {ENTROPY_W_PATH} 없음 → fallback 가중치 사용")
    ENTROPY_W = {
        "elderly_res_ratio": 0.465370,
        "vuln_peak_pop":     0.362159,
        "grid_congestion":   0.067276,
        "grid_avg_speed":    0.048505,
        "weekend_ratio":     0.032431,
        "gap_cnt":           0.017510,
        "vuln_float_ratio":  0.006749,
    }

def _minmax_norm(values):
    """None·빈문자열 포함 리스트를 0~1 정규화 (결측은 0 처리)"""
    nums = [float(v) for v in values if v is not None and v != ""]
    if not nums:
        return [0.0] * len(values)
    vmin, vmax = min(nums), max(nums)
    denom = vmax - vmin
    if denom == 0:
        return [0.5 if (v is not None and v != "") else 0.0 for v in values]
    return [
        (float(v) - vmin) / denom if (v is not None and v != "") else 0.0
        for v in values
    ]

# 인자별 전체값 추출 → 정규화
raw_vals = {k: [r.get(k, "") for r in result] for k in ENTROPY_W}
norm_vals = {k: _minmax_norm(raw_vals[k]) for k in ENTROPY_W}

# 각 격자에 엔트로피 가중 보정지수 + 새 종합 위험 지수 산출
for i, r in enumerate(result):
    correction_index = sum(ENTROPY_W[k] * norm_vals[k][i] for k in ENTROPY_W)
    r["correction_index"]       = round(correction_index, 6)
    r["entropy_composite_risk"] = round(r["epdo_total"] * (1 + correction_index), 2)

# 엔트로피 기준 재정렬 + 순위
result.sort(key=lambda x: -x["entropy_composite_risk"])
for i, r in enumerate(result, 1):
    r["entropy_rank"] = i

print(f"    엔트로피 가중치 적용 완료: {len(result):,}개 격자")
print(f"    가중치 구성: 노인거주({ENTROPY_W['elderly_res_ratio']:.1%}) "
      f"| 취약시간({ENTROPY_W['vuln_peak_pop']:.1%}) "
      f"| 혼잡({ENTROPY_W['grid_congestion']:.1%}) ...")

## ── 12. 저장

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
cols = list(result[0].keys())
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(result)

## ── 13. 하남교산 예측 위험도 — 엔트로피 기준으로 전이

In [ ]:
print("\n[12] 하남교산 예측 위험도 산출 중...")

# 4개 신도시에서 blockType별 평균 entropy_composite_risk 계산
lu_risk = defaultdict(list)
for r in result:
    if r["land_use"] != "미분류":
        lu_risk[r["land_use"]].append(r["entropy_composite_risk"])
lu_avg_risk = {lt: round(sum(v) / len(v), 2) for lt, v in lu_risk.items()}

# 등급 기준: lu_avg_risk 분포의 75·25 퍼센타일 사용
all_risks   = [r["entropy_composite_risk"] for r in result]
overall_avg = round(sum(all_risks) / len(all_risks), 2)
lu_vals     = sorted(lu_avg_risk.values())
def _pct(vals, p):
    idx = (p / 100) * (len(vals) - 1)
    lo, hi = int(idx), min(int(idx) + 1, len(vals) - 1)
    return vals[lo] + (idx - lo) * (vals[hi] - vals[lo])
grade_high = _pct(lu_vals, 75)   # 상위 25% → 고위험
grade_mid  = _pct(lu_vals, 25)   # 중위 50% → 중위험
print(f"    등급 기준 — 고위험(75%ile): {grade_high:.1f} | "
      f"중위험(25~75%ile) | 저위험(<25%ile): {grade_mid:.1f}")

# 23번 하남교산 토지이용 로드
hanam_gdf   = gpd.read_file(LU_HANAM)
hanam_result = []
for _, feat in hanam_gdf.iterrows():
    btype    = feat.get("blockType", "미분류") or "미분류"
    btype    = btype.strip().replace("\r\n", "").replace("\n", "")
    # 동일 blockType 없으면 카테고리로 매칭
    if btype in lu_avg_risk:
        pred_risk = lu_avg_risk[btype]
        basis = "동일유형"
    elif btype in SCHOOL_TYPES:
        matched = [lu_avg_risk[k] for k in SCHOOL_TYPES if k in lu_avg_risk]
        pred_risk = round(sum(matched) / len(matched), 2) if matched else overall_avg
        basis = "학교유형평균"
    elif btype in RESIDENT_TYPES:
        matched = [lu_avg_risk[k] for k in RESIDENT_TYPES if k in lu_avg_risk]
        pred_risk = round(sum(matched) / len(matched), 2) if matched else overall_avg
        basis = "주거유형평균"
    elif btype in COMMERCIAL_TYPES:
        matched = [lu_avg_risk[k] for k in COMMERCIAL_TYPES if k in lu_avg_risk]
        pred_risk = round(sum(matched) / len(matched), 2) if matched else overall_avg
        basis = "상업유형평균"
    else:
        pred_risk = overall_avg
        basis = "전체평균"

    # 위험 등급 분류 (4개 신도시 분포의 75·25 퍼센타일 기준)
    if pred_risk >= grade_high:
        risk_grade = "고위험"
    elif pred_risk >= grade_mid:
        risk_grade = "중위험"
    else:
        risk_grade = "저위험"

    hanam_result.append({
        "zoneCode":       feat.get("zoneCode", ""),
        "zoneName":       feat.get("zoneName", ""),
        "blockName":      feat.get("blockName", ""),
        "blockType":      btype,
        "pred_risk":      pred_risk,
        "risk_grade":     risk_grade,
        "basis":          basis,
    })

hanam_result.sort(key=lambda x: -x["pred_risk"])
with open(HANAM_OUTPUT, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(hanam_result[0].keys()))
    w.writeheader()
    w.writerows(hanam_result)

hanam_high = sum(1 for r in hanam_result if r["risk_grade"] == "고위험")
hanam_mid  = sum(1 for r in hanam_result if r["risk_grade"] == "중위험")
hanam_low  = sum(1 for r in hanam_result if r["risk_grade"] == "저위험")
print(f"    하남교산 전체 구역: {len(hanam_result):,}개")
print(f"    고위험: {hanam_high}개 | 중위험: {hanam_mid}개 | 저위험: {hanam_low}개")

## ── 결과 요약

In [ ]:
print(f"\n{'='*60}")
print(f"[최종 결과]")
print(f"    분석 격자: {len(result):,}개")

label_cnt = Counter()
for r in result:
    for lb in r["risk_label"].split(" | "):
        if lb:
            label_cnt[lb] += 1
print(f"\n    위험 특성 분포 (중앙값 기준 개선):")
for lb, cnt in label_cnt.most_common():
    print(f"      {lb:15s}: {cnt:5d}개 ({cnt/len(result)*100:.1f}%)")

print(f"\n    종합 위험 지수 상위 10개 (엔트로피 가중치 적용):")
print(f"  {'순위':>4} {'격자ID':12s} {'EPDO':>6} "
      f"{'보정지수':>8} {'기존위험':>9} {'엔트로피위험':>12}  라벨")
print("  " + "-" * 90)
for r in result[:10]:
    print(f"  {r['entropy_rank']:>4} {r['grid_gid']:12s} {r['epdo_total']:>6.0f} "
          f"{r['correction_index']:>8.4f} {r['composite_risk']:>9.1f} "
          f"{r['entropy_composite_risk']:>12.1f}  {r['risk_label']}")

print(f"\n    하남교산 고위험 구역 상위 5개:")
print(f"  {'블록명':15s} {'유형':12s} {'예측위험':>8} {'등급':>5}")
print("  " + "-" * 50)
for r in hanam_result[:5]:
    print(f"  {str(r['blockName']):15s} {str(r['blockType']):12s} "
          f"{r['pred_risk']:>8.1f} {r['risk_grade']:>5}")

print(f"\n저장: {OUTPUT_PATH}")
print(f"저장: {HANAM_OUTPUT}")
print("=" * 60)

---

## 09_ENTROPY_WEIGHT

# STEP 09 - 엔트로피 가중법으로 보정 계수 가중치 산출

- 입력: epdo_analysis/output/08_격자_종합위험지수.csv
- 출력: epdo_analysis/output/09_엔트로피_가중치.csv

엔트로피 가중법 (Entropy Weight Method):
  변동이 클수록 정보량이 많다 → 가중치 높음
  변동이 작을수록 정보량이 없다 → 가중치 낮음

4단계:
  1. 정규화: 각 인자를 0~1 범위로 통일
  2. 비율 계산: 각 격자의 인자가 전체에서 차지하는 비중
  3. 엔트로피 계산: 값이 균일할수록 엔트로피 → 1 (정보 없음)
  4. 가중치 산출: (1 - 엔트로피) 정규화

In [ ]:
import csv
import math
import os

INPUT_PATH  = os.path.join(EPDO_DIR, "epdo_analysis", "output", "08_격자_종합위험지수.csv")
OUTPUT_PATH = os.path.join(EPDO_DIR, "epdo_analysis", "output", "09_엔트로피_가중치.csv")

# ── 가중치를 산출할 인자 목록 ─────────────────────────────────────────────────
# composite_risk 공식의 7개 보정 인자
FACTORS = {
    "elderly_res_ratio":  "노인거주비율     (03 거주인구)",
    "vuln_float_ratio":   "교통약자유동비율  (04 유동인구)",
    "gap_cnt":            "인프라공백수     (STEP 06)",
    "grid_avg_speed":     "격자평균속도     (09 속도)",
    "grid_congestion":    "혼잡위험도       (11·12 혼잡강도)",
    "vuln_peak_pop":      "취약시간대인구   (05·06 시간대인구)",
    "weekend_ratio":      "주말방문비율     (07 서비스인구)",
}

FACTOR_KEYS = list(FACTORS.keys())


# ── 통계 유틸 ─────────────────────────────────────────────────────────────────

def normalize_minmax(values):
    """최솟값-최댓값 정규화 → 0~1"""
    v_min = min(values)
    v_max = max(values)
    denom = v_max - v_min
    if denom == 0:
        return [0.5] * len(values)   # 모두 같으면 0.5로 처리
    return [(v - v_min) / denom for v in values]


def entropy_weight(matrix):
    """
    matrix: dict { factor_key: [값1, 값2, ..., 값n] }
    반환  : dict { factor_key: weight }
    """
    n = len(next(iter(matrix.values())))   # 격자 수
    k = 1.0 / math.log(n)                 # 정규화 상수

    entropies = {}
    for key, values in matrix.items():
        norm = normalize_minmax(values)

        # 비율 계산 (0 방지를 위해 tiny 값 추가)
        total = sum(norm) or 1.0
        p = [v / total for v in norm]

        # 엔트로피: ej = -k × Σ(pij × ln(pij))
        e = 0.0
        for pi in p:
            if pi > 0:
                e += pi * math.log(pi)
        entropies[key] = -k * e

    # 편차 dj = 1 - ej
    deviations = {k: 1.0 - e for k, e in entropies.items()}

    # 가중치 wj = dj / Σdj
    total_dev = sum(deviations.values())
    weights = {k: round(d / total_dev, 6) for k, d in deviations.items()}

    return entropies, deviations, weights


# ── 메인 ─────────────────────────────────────────────────────────────────────

print("=" * 60)
print("STEP 09 - 엔트로피 가중법 가중치 산출")
print("=" * 60)

# 1. 데이터 로드
print("\n[1] 데이터 로드 중...")
rows = []
with open(INPUT_PATH, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        rows.append(row)
print(f"    격자 수: {len(rows):,}개")

# 2. 인자별 값 추출 (빈 값은 0 또는 평균 대체)
print("\n[2] 인자별 데이터 추출 중...")
matrix = {}
missing_info = {}

for key in FACTOR_KEYS:
    raw = []
    missing = 0
    for row in rows:
        val = row.get(key, "")
        if val == "" or val is None:
            missing += 1
            raw.append(None)
        else:
            try:
                raw.append(float(val))
            except ValueError:
                missing += 1
                raw.append(None)

    # 결측값 처리: 비결측 평균으로 대체
    valid = [v for v in raw if v is not None]
    avg = sum(valid) / len(valid) if valid else 0.0
    filled = [v if v is not None else avg for v in raw]

    matrix[key] = filled
    missing_info[key] = missing

for key, miss in missing_info.items():
    pct = miss / len(rows) * 100
    fill_note = f"(결측 {miss}개 → 평균 {sum(v for v in matrix[key])/len(matrix[key]):.4f}으로 대체)" if miss > 0 else ""
    print(f"    {key:25s}: {len(rows)-miss:4d}개 유효  {fill_note}")

# 3. 엔트로피 가중치 계산
print("\n[3] 엔트로피 가중치 계산 중...")
entropies, deviations, weights = entropy_weight(matrix)

# 가중치 기준 내림차순 정렬
sorted_keys = sorted(weights, key=lambda k: -weights[k])

# 4. 결과 출력
print("\n" + "=" * 60)
print("  [결과] 인자별 엔트로피 가중치")
print("=" * 60)
print(f"\n  {'순위':>3}  {'인자':25s}  {'엔트로피':>10}  {'편차':>8}  {'가중치':>8}  설명")
print("  " + "-" * 80)

result = []
for rank, key in enumerate(sorted_keys, 1):
    desc = FACTORS[key]
    e    = entropies[key]
    d    = deviations[key]
    w    = weights[key]
    bar  = "█" * int(w * 100)

    print(f"  {rank:>3}. {key:25s}  {e:10.6f}  {d:8.6f}  {w:8.4%}  {desc}")
    print(f"       {bar}")

    result.append({
        "순위":   rank,
        "인자":   key,
        "설명":   desc,
        "엔트로피": round(e, 6),
        "편차":   round(d, 6),
        "가중치": round(w, 6),
        "가중치(%)": f"{w*100:.2f}%",
    })

print()

# 5. 현재 공식 vs 가중치 적용 공식 비교
print("=" * 60)
print("  [현재 공식] — 모든 인자 동등 가중 (곱셈)")
print("=" * 60)
print("""
  composite_risk = EPDO
× vuln_correction    (노인거주 + 교통약자유동)
× infra_penalty      (인프라공백)
× speed_weight       (속도)
× congestion_factor  (혼잡강도)
× peak_factor        (취약시간 인구)
× weekend_factor     (주말비율)
""")

print("=" * 60)
print("  [가중치 적용 공식] — 엔트로피 가중합")
print("=" * 60)

# 가중합 방식으로 변환
factor_map = {
    "elderly_res_ratio": "elderly_res_ratio",
    "vuln_float_ratio":  "vuln_float_ratio",
    "gap_cnt":           "gap_cnt / 9",
    "grid_avg_speed":    "grid_avg_speed / 60",
    "grid_congestion":   "grid_congestion / 100",
    "vuln_peak_pop":     "peak_factor - 1",
    "weekend_ratio":     "weekend_ratio",
}

print("\n  보정지수 = Σ(가중치 × 정규화된 인자값)")
print()
for key in sorted_keys:
    w = weights[key]
    expr = factor_map.get(key, key)
    print(f"    {w:.4%} × {expr:35s}  ({FACTORS[key]})")

print("""
  composite_risk = EPDO × (1 + 보정지수)

  ※ 1을 더하는 이유: EPDO가 0인 격자도 보정값이 0보다 크게 만들기 위함
""")

# 6. 해석 안내
print("=" * 60)
print("  [해석 가이드]")
print("=" * 60)
top3 = sorted_keys[:3]
low3 = sorted_keys[-3:]
print(f"\n  가중치 상위 3개 (가장 중요한 인자):")
for k in top3:
    print(f"    → {k}: {weights[k]:.4%}  ─  {FACTORS[k]}")
print(f"\n  가중치 하위 3개 (덜 중요한 인자):")
for k in low3:
    print(f"    → {k}: {weights[k]:.4%}  ─  {FACTORS[k]}")

print("""
  ※ 가중치가 높다 = 격자마다 값이 많이 달랐다 = 차별화 능력이 높다
  ※ 가중치가 낮다 = 격자마다 값이 비슷했다  = 차별화 능력이 낮다
  ※ 이는 "더 중요한 인자"가 아닌 "더 많이 변동하는 인자"임을 주의
 전문가 판단(AHP)과 병행하면 더 신뢰성 높은 가중치 산출 가능
""")

# 7. CSV 저장
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
cols = list(result[0].keys())
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(result)

print(f"저장: {OUTPUT_PATH}")
print("=" * 60)

---

## 10_NB_REGRESSION_WEIGHT

# STEP 10 - 음이항 회귀 주축 + 엔트로피/상관계수 3중 교차검증 통합 가중치

목적:
  엔트로피 단독의 한계(변별력 ≠ 중요도)를 보완하기 위해
  음이항 회귀(NB), EPDO 상관계수(Corr), 엔트로피(Entropy) 3가지 방법으로
  각 인자의 가중치를 산출하고, 기하평균으로 통합한다.

각 방법의 역할:
  - NB 회귀    : "이 인자가 사고 건수에 실제로 얼마나 영향을 주나?" (주축)
  - Corr 가중  : "이 인자가 사고 심각도(EPDO)와 얼마나 연동되나?" (교차검증 1)
  - 엔트로피   : "이 인자가 격자를 얼마나 잘 구분하나?"           (교차검증 2)

통합 방식:
  w_final = geometric_mean(w_nb, w_corr, w_entropy) → 정규화

입력: epdo_analysis/output/08_격자_종합위험지수.csv
      epdo_analysis/output/09_엔트로피_가중치.csv
출력: epdo_analysis/output/10_통합가중치_최종.csv

In [ ]:
import csv
import math
import os

import numpy as np
import pandas as pd

# statsmodels 음이항 회귀 (없으면 Poisson fallback)
try:
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except ImportError:
    from sklearn.linear_model import PoissonRegressor
    from sklearn.preprocessing import StandardScaler
    HAS_STATSMODELS = False

INPUT_08      = os.path.join(EPDO_DIR, "epdo_analysis", "output", "08_격자_종합위험지수.csv")
INPUT_09      = os.path.join(EPDO_DIR, "epdo_analysis", "output", "09_엔트로피_가중치.csv")
OUTPUT_PATH   = os.path.join(EPDO_DIR, "epdo_analysis", "output", "10_통합가중치_최종.csv")

FACTORS = {
    "elderly_res_ratio": "노인거주비율",
    "vuln_float_ratio":  "교통약자유동비율",
    "gap_cnt":           "인프라공백수",
    "grid_avg_speed":    "격자평균속도",
    "grid_congestion":   "혼잡위험도",
    "vuln_peak_pop":     "취약시간대인구",
    "weekend_ratio":     "주말방문비율",
}
FACTOR_KEYS = list(FACTORS.keys())

# grid_avg_speed 는 사고와 반대 방향 → 절댓값 처리 명시
NEGATIVE_EXPECTED = {"grid_avg_speed"}


# ── 유틸 ──────────────────────────────────────────────────────────────────────

def normalize(d: dict) -> dict:
    """dict 값을 합계=1로 정규화"""
    total = sum(d.values())
    return {k: v / total for k, v in d.items()}


def geometric_mean_weights(*weight_dicts):
    """
    여러 가중치 dict의 기하평균 → 정규화
    기하평균: 각 인자마다 모든 방법 가중치를 곱해 n제곱근 취함
    """
    n = len(weight_dicts)
    keys = list(weight_dicts[0].keys())
    raw = {}
    for k in keys:
        product = 1.0
        for wd in weight_dicts:
            product *= wd[k]
        raw[k] = product ** (1.0 / n)
    return normalize(raw)


def spearman_corr(rank_a, rank_b):
    """두 순위 리스트의 스피어만 상관계수"""
    n = len(rank_a)
    d2 = sum((a - b) ** 2 for a, b in zip(rank_a, rank_b))
    return 1 - (6 * d2) / (n * (n ** 2 - 1))


# ── 1단계: 데이터 로드 ─────────────────────────────────────────────────────────

def load_data():
    df = pd.read_csv(INPUT_08, encoding="utf-8-sig")
    print(f"    총 격자 수: {len(df):,}개")

    # 결측값 처리: 비결측 평균으로 대체
    cols_needed = FACTOR_KEYS + ["accident_cnt", "epdo_total"]
    for col in cols_needed:
        if col in df.columns:
            missing = df[col].isna().sum()
            if missing > 0:
                df[col] = df[col].fillna(df[col].mean())
                print(f"    {col}: 결측 {missing}개 → 평균 대체")
        else:
            print(f"    [경고] {col} 컬럼 없음 → 0으로 채움")
            df[col] = 0.0

    # accident_cnt는 정수형
    df["accident_cnt"] = df["accident_cnt"].clip(lower=0).round().astype(int)
    return df


# ── 2단계: 음이항 회귀 → NB 가중치 ───────────────────────────────────────────

def nb_regression_weights(df):
    X_raw = df[FACTOR_KEYS].values
    y     = df["accident_cnt"].values

    # 표준화 (StandardScaler): 계수가 변수 간 직접 비교 가능해짐
    mu    = X_raw.mean(axis=0)
    sigma = X_raw.std(axis=0)
    sigma[sigma == 0] = 1.0
    X_std = (X_raw - mu) / sigma

    if HAS_STATSMODELS:
        print("    [모델] statsmodels 음이항(NegativeBinomial) 회귀")
        X_sm = sm.add_constant(X_std)
        model = sm.NegativeBinomial(y, X_sm)
        result = model.fit(disp=False, maxiter=200)
        coefs = result.params[1:]          # 상수항 제외
        pvals = result.pvalues[1:]
        model_name = "음이항 회귀 (NegativeBinomial)"
    else:
        print("    [모델] sklearn PoissonRegressor (statsmodels 미설치)")
        from sklearn.linear_model import PoissonRegressor
        reg = PoissonRegressor(max_iter=500)
        reg.fit(X_std, y)
        coefs = reg.coef_
        pvals = [None] * len(coefs)
        model_name = "푸아송 회귀 (Poisson, NB 근사)"

    # 표준화 계수 절댓값 → 정규화
    abs_coefs = {k: abs(float(coefs[i])) for i, k in enumerate(FACTOR_KEYS)}
    weights   = normalize(abs_coefs)

    return weights, abs_coefs, pvals, model_name


# ── 3단계: EPDO 상관계수 → Corr 가중치 ───────────────────────────────────────

def corr_weights(df):
    corrs = {}
    for key in FACTOR_KEYS:
        c = df[key].corr(df["epdo_total"])
        corrs[key] = abs(c)

    weights = normalize(corrs)
    return weights, corrs


# ── 4단계: 엔트로피 가중치 로드 ───────────────────────────────────────────────

def load_entropy_weights():
    entropy_w = {}
    with open(INPUT_09, "r", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            key = row["인자"]
            if key in FACTOR_KEYS:
                entropy_w[key] = float(row["가중치"])
    # 혹시 누락된 인자 있으면 균등값으로 채움
    for k in FACTOR_KEYS:
        if k not in entropy_w:
            entropy_w[k] = 1.0 / len(FACTOR_KEYS)
    return normalize(entropy_w)


# ── 5단계: 합의도(★) 판정 ────────────────────────────────────────────────────

def consensus_star(nb_rank, corr_rank, ent_rank, n=7):
    """상위 절반(top 4/7) 기준 합의도"""
    threshold = n // 2 + 1   # 4
    top_count = sum([
        nb_rank   <= threshold,
        corr_rank <= threshold,
        ent_rank  <= threshold,
    ])
    return "★★★" if top_count == 3 else "★★" if top_count == 2 else "★"


# ── 메인 ─────────────────────────────────────────────────────────────────────

sep = "=" * 70
print(sep)
print("STEP 10 - 3중 교차검증 통합 가중치 산출")
print(sep)

# 1. 데이터 로드
print("\n[1] 데이터 로드")
df = load_data()

# 2. 음이항 회귀
print("\n[2] 음이항 회귀 가중치 계산")
nb_w, nb_coefs, pvals, model_name = nb_regression_weights(df)
print(f"    사용 모델: {model_name}")

# 3. 상관계수 가중치
print("\n[3] EPDO 상관계수 가중치 계산")
corr_w, corrs = corr_weights(df)
for k in FACTOR_KEYS:
    direction = "↑" if corrs[k] >= 0 else "↓"
    print(f"    {k:25s}: r={df[k].corr(df['epdo_total']):+.4f} {direction}  → |r|={corrs[k]:.4f}")

# 4. 엔트로피 가중치
print("\n[4] 엔트로피 가중치 로드")
ent_w = load_entropy_weights()
for k in FACTOR_KEYS:
    print(f"    {k:25s}: {ent_w[k]:.6f}")

# 5. 통합 가중치 (기하평균)
print("\n[5] 통합 가중치 계산 (기하평균)")
final_w = geometric_mean_weights(nb_w, corr_w, ent_w)

# 6. 순위 산출
nb_rank   = {k: sorted(nb_w,   key=lambda x: -nb_w[x]).index(k)  + 1 for k in FACTOR_KEYS}
corr_rank = {k: sorted(corr_w, key=lambda x: -corr_w[x]).index(k)+ 1 for k in FACTOR_KEYS}
ent_rank  = {k: sorted(ent_w,  key=lambda x: -ent_w[x]).index(k) + 1 for k in FACTOR_KEYS}
final_rank= {k: sorted(final_w,key=lambda x: -final_w[x]).index(k)+1 for k in FACTOR_KEYS}

# 7. 결과 출력
print("\n" + sep)
print("  [결과] 3중 교차검증 통합 가중치")
print(sep)
header = f"  {'인자':25s}  {'NB회귀':>8}  {'EPDO상관':>8}  {'엔트로피':>8}  {'통합':>8}  합의도  설명"
print(f"\n{header}")
print("  " + "-" * 85)

sorted_keys = sorted(FACTOR_KEYS, key=lambda k: -final_w[k])
result_rows = []

for k in sorted_keys:
    star = consensus_star(nb_rank[k], corr_rank[k], ent_rank[k])
    desc = FACTORS[k]
    pval_str = f"p={float(pvals[FACTOR_KEYS.index(k)]):.3f}" if pvals[0] is not None else ""
    print(
        f"  {k:25s}  {nb_w[k]:8.4%}  {corr_w[k]:8.4%}  {ent_w[k]:8.4%}"
        f"  {final_w[k]:8.4%}  {star:4s}  {desc}"
    )
    result_rows.append({
        "최종순위":     final_rank[k],
        "인자":         k,
        "설명":         desc,
        "NB회귀_가중치":   round(nb_w[k],  6),
        "NB회귀_순위":     nb_rank[k],
        "EPDO상관_가중치": round(corr_w[k], 6),
        "EPDO상관_순위":   corr_rank[k],
        "엔트로피_가중치": round(ent_w[k],  6),
        "엔트로피_순위":   ent_rank[k],
        "통합가중치":    round(final_w[k], 6),
        "합의도":        star,
    })

# 8. 스피어만 순위 상관 (방법 간 일치도)
print(f"\n{'─'*50}")
print("  [방법 간 스피어만 순위 상관계수]")
nb_r   = [nb_rank[k]   for k in FACTOR_KEYS]
corr_r = [corr_rank[k] for k in FACTOR_KEYS]
ent_r  = [ent_rank[k]  for k in FACTOR_KEYS]
sc_nb_corr = spearman_corr(nb_r, corr_r)
sc_nb_ent  = spearman_corr(nb_r, ent_r)
sc_corr_ent= spearman_corr(corr_r, ent_r)
print(f"  NB회귀 ↔ EPDO상관:  {sc_nb_corr:+.4f}")
print(f"  NB회귀 ↔ 엔트로피:  {sc_nb_ent:+.4f}")
print(f"  EPDO상관 ↔ 엔트로피: {sc_corr_ent:+.4f}")

# 9. 핵심 변수 요약
core    = [k for k in sorted_keys if consensus_star(nb_rank[k], corr_rank[k], ent_rank[k]) == "★★★"]
major   = [k for k in sorted_keys if consensus_star(nb_rank[k], corr_rank[k], ent_rank[k]) == "★★"]
ref     = [k for k in sorted_keys if consensus_star(nb_rank[k], corr_rank[k], ent_rank[k]) == "★"]

print(f"\n{'─'*50}")
print("  [합의도 기반 변수 분류]")
print(f"\n  ★★★ 핵심변수 (3가지 방법 모두 상위 4위 이내):")
for k in core:
    print(f"    → {k}: 통합 {final_w[k]:.2%}  ({FACTORS[k]})")
print(f"\n  ★★  주요변수 (2가지 방법 상위 4위 이내):")
for k in major:
    print(f"    → {k}: 통합 {final_w[k]:.2%}  ({FACTORS[k]})")
print(f"\n  ★   참고변수 (1가지 이하):")
for k in ref:
    print(f"    → {k}: 통합 {final_w[k]:.2%}  ({FACTORS[k]})")

# 10. 엔트로피 단독 대비 변화
print(f"\n{'─'*50}")
print("  [엔트로피 단독 → 통합 가중치 순위 변화]")
ent_sorted  = sorted(FACTOR_KEYS, key=lambda k: -ent_w[k])
final_sorted= sorted(FACTOR_KEYS, key=lambda k: -final_w[k])
for k in FACTOR_KEYS:
    old = ent_rank[k]
    new = final_rank[k]
    delta = old - new
    arrow = f"▲{delta}" if delta > 0 else f"▼{abs(delta)}" if delta < 0 else "─"
    print(f"  {k:25s}: 엔트로피 {old}위 → 통합 {new}위  {arrow}")

print(f"\n  통합 가중치 합계: {sum(final_w.values()):.6f}  (1.0 검증)")
print()

# 11. CSV 저장
result_rows.sort(key=lambda r: r["최종순위"])
cols = list(result_rows[0].keys())
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(result_rows)

print(f"저장: {OUTPUT_PATH}")
print(sep)

---

## 11_FACILITY_EFFECT

# STEP 11: 스마트 안전시설물 설치 효과 계량화

=============================================
문헌 기반 시설물별 사고 감소율을 적용하여
고위험 격자별 시설물 설치 후 예상 EPDO 감소량을 산출한다.

입력:
  epdo_analysis/output/06_인프라공백_고위험격자.csv
  epdo_analysis/output/08_격자_종합위험지수.csv   (entropy_rank 참조)

출력:
  epdo_analysis/output/11_시설물설치_효과예측.csv

방법론:
  - 문헌 기반 보수적 감소율 적용 (실증 전후 비교 데이터 부재 시 표준 접근법)
  - 독립 효과 가정: 총 감소율 = 1 - Π(1 - r_i)
  - 효율성 = EPDO 감소량 / 공백 시설물 수 (시설물 1개당 기대 효과)
  - 설치 우선순위 = 효율성 기준 내림차순

출처 근거 (보수적 하한값 적용):
  - 횡단보도 15%: 교통안전공단 (보행자 사고 15~25% 감소)
  - 어린이보호구역 25%: 행정안전부 (어린이 사고 30~40% 감소)
  - 과속방지턱 20%: 도로교통공단 (사고 20~30% 감소)
  - CCTV 개소 12%: 경찰청 (무인 단속 효과 10~20%)
  - CCTV 대수 5%: CCTV 개소와 중복 효과 최소화
  - 버스정류장 8%: 보행자 대기 공간 정비 효과

In [ ]:
import csv
import os

# ── 경로 설정 ─────────────────────────────────────────────────────────────────
INPUT_GAP   = os.path.join(EPDO_DIR, "epdo_analysis", "output", "06_인프라공백_고위험격자.csv")
INPUT_RISK  = os.path.join(EPDO_DIR, "epdo_analysis", "output", "08_격자_종합위험지수.csv")
OUTPUT_PATH = os.path.join(EPDO_DIR, "epdo_analysis", "output", "11_시설물설치_효과예측.csv")

# ── 시설물별 문헌 기반 보수적 사고 감소율 ──────────────────────────────────────
REDUCTION_RATES = {
    "crosswalk_cnt":  0.15,   # 횡단보도: 교통안전공단 15~25% → 하한 15%
    "child_zone_cnt": 0.25,   # 어린이보호구역: 행안부 30~40% → 하한 25%
    "speedbump_cnt":  0.20,   # 과속방지턱: 도로교통공단 20~30% → 하한 20%
    "cctv_cnt":       0.12,   # CCTV 개소: 경찰청 10~20% → 하한 12%
    "cctv_cam_cnt":   0.05,   # CCTV 대수: 개소 효과와 중복 최소화 → 5%
    "bus_stop_cnt":   0.08,   # 버스정류장: 보행자 대기 공간 정비 → 8%
}

FACILITY_KR = {
    "crosswalk_cnt":  "횡단보도",
    "child_zone_cnt": "어린이보호구역",
    "speedbump_cnt":  "과속방지턱",
    "cctv_cnt":       "CCTV(개소)",
    "cctv_cam_cnt":   "CCTV(대수)",
    "bus_stop_cnt":   "버스정류장",
}


def parse_gap_items(gap_str):
    """gap_items 문자열에서 공백 시설물 목록을 추출.
    예: "crosswalk_cnt(없음) | cctv_cnt(없음)" → {'crosswalk_cnt', 'cctv_cnt'}
    """
    if not gap_str or gap_str.strip() == "":
        return set()
    missing = set()
    for item in gap_str.split("|"):
        item = item.strip()
        if "(" in item:
            col = item.split("(")[0].strip()
            if col in REDUCTION_RATES:
                missing.add(col)
    return missing


def combined_reduction(missing_facilities):
    """독립 효과 가정: 총 감소율 = 1 - Π(1 - r_i)"""
    survive = 1.0
    for fac in missing_facilities:
        r = REDUCTION_RATES.get(fac, 0.0)
        survive *= (1.0 - r)
    return round(1.0 - survive, 6)

## ── 1. 08번 파일에서 entropy_rank 로드

In [ ]:
print("[1] entropy_rank 로드 중...")
entropy_rank_map = {}
entropy_risk_map = {}
with open(INPUT_RISK, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        gid = row["grid_gid"]
        entropy_rank_map[gid] = row.get("entropy_rank", "")
        entropy_risk_map[gid] = row.get("entropy_composite_risk", "")
print(f"    로드 완료: {len(entropy_rank_map):,}개 격자")

## ── 2. 06번 공백 파일 로드 및 효과 계산

In [ ]:
print("\n[2] 시설물 설치 효과 계산 중...")
results = []
facility_epdo_saved = {fac: 0.0 for fac in REDUCTION_RATES}

with open(INPUT_GAP, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    for row in reader:
        gid       = row["grid_gid"]
        epdo_before = float(row.get("epdo_total", 0) or 0)
        acc_cnt   = int(row.get("accident_cnt", 0) or 0)
        death_cnt = int(row.get("사망_cnt", 0) or 0)
        injury_cnt= int(row.get("중상_cnt", 0) or 0)
        gap_cnt   = int(row.get("gap_cnt", 0) or 0)
        gap_items = row.get("gap_items", "")

        # 공백 시설물 파싱
        missing = parse_gap_items(gap_items)

        # 총 감소율 산출
        total_reduction = combined_reduction(missing)

        # 설치 후 예상 EPDO
        epdo_after = round(epdo_before * (1.0 - total_reduction), 2)
        epdo_saved = round(epdo_before - epdo_after, 2)

        # 효율성: 시설물 1개당 EPDO 감소량
        efficiency = round(epdo_saved / gap_cnt, 4) if gap_cnt > 0 else 0.0

        # 시설물별 기여 감소량 (단순 독립 분배: 각 시설의 단독 감소량)
        for fac in missing:
            r = REDUCTION_RATES.get(fac, 0)
            fac_saved = epdo_before * r
            facility_epdo_saved[fac] += fac_saved

        results.append({
            "grid_gid":            gid,
            "epdo_before":         epdo_before,
            "accident_cnt":        acc_cnt,
            "사망_cnt":            death_cnt,
            "중상_cnt":            injury_cnt,
            "gap_cnt":             gap_cnt,
            "missing_facilities":  " | ".join(sorted(missing)),
            "total_reduction_pct": round(total_reduction * 100, 2),
            "epdo_after":          epdo_after,
            "epdo_saved":          epdo_saved,
            "efficiency":          efficiency,
            "entropy_rank":        entropy_rank_map.get(gid, ""),
            "entropy_risk":        entropy_risk_map.get(gid, ""),
        })

print(f"    계산 완료: {len(results):,}개 격자")

## ── 3. 설치 우선순위: 효율성(시설물 1개당 EPDO 감소) 기준

In [ ]:
print("\n[3] 설치 우선순위 산출 중...")
results.sort(key=lambda x: x["efficiency"], reverse=True)
for i, row in enumerate(results, 1):
    row["install_priority"] = i

## ── 4. CSV 저장

In [ ]:
print("\n[4] CSV 저장 중...")
fieldnames = [
    "install_priority", "grid_gid", "epdo_before", "accident_cnt",
    "사망_cnt", "중상_cnt", "gap_cnt", "missing_facilities",
    "total_reduction_pct", "epdo_after", "epdo_saved", "efficiency",
    "entropy_rank", "entropy_risk",
]
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)
print(f"    저장 완료: {OUTPUT_PATH}")

## ── 5. 집계 요약 출력

In [ ]:
print("\n" + "="*60)
print("  STEP 11 결과 요약")
print("="*60)

total_epdo_before = sum(r["epdo_before"] for r in results)
total_epdo_saved  = sum(r["epdo_saved"]  for r in results)
total_grids       = len(results)
avg_reduction     = total_epdo_saved / total_epdo_before * 100 if total_epdo_before > 0 else 0

print(f"\n  분석 대상 격자 수 : {total_grids:,}개")
print(f"  현재 총 EPDO     : {total_epdo_before:,.0f}점")
print(f"  설치 후 예상 EPDO: {(total_epdo_before - total_epdo_saved):,.0f}점")
print(f"  총 예상 감소량   : {total_epdo_saved:,.0f}점 ({avg_reduction:.1f}%)")

print("\n  [시설물 유형별 총 기여 감소량]")
fac_sorted = sorted(facility_epdo_saved.items(), key=lambda x: x[1], reverse=True)
for fac, saved in fac_sorted:
    pct = saved / total_epdo_before * 100 if total_epdo_before > 0 else 0
    print(f"    {FACILITY_KR.get(fac, fac):12s}: {saved:8,.0f}점 ({pct:.1f}%)")

print("\n  [효율성 기준 설치 우선순위 Top-10]")
print(f"  {'순위':>4} {'격자ID':12} {'현재EPDO':>8} {'감소율':>6} {'EPDO감소':>8} {'효율성':>8} {'entropy순위':>8}")
for r in results[:10]:
    print(f"  {r['install_priority']:>4} {r['grid_gid']:12} "
          f"{r['epdo_before']:>8.0f} {r['total_reduction_pct']:>5.1f}% "
          f"{r['epdo_saved']:>8.0f} {r['efficiency']:>8.2f} "
          f"{r['entropy_rank']:>8}")

print("\n  [EPDO 절대 감소량 기준 Top-10]")
epdo_sorted = sorted(results, key=lambda x: x["epdo_saved"], reverse=True)
print(f"  {'순위':>4} {'격자ID':12} {'현재EPDO':>8} {'감소율':>6} {'EPDO감소':>8} {'entropy순위':>8}")
for i, r in enumerate(epdo_sorted[:10], 1):
    print(f"  {i:>4} {r['grid_gid']:12} "
          f"{r['epdo_before']:>8.0f} {r['total_reduction_pct']:>5.1f}% "
          f"{r['epdo_saved']:>8.0f} {r['entropy_rank']:>8}")

print("="*60)